# Notebook de Révision — DevOps, MLOps & FastAPI BLOC 3

> Dataset fictif TechShop. Toutes les commandes shell sont illustrées comme chaînes.

## Sommaire
1. [Notebooks Jupyter](#1-jupyter)
2. [Variables d'environnement — export & .bashrc](#2-env-bash)
3. [Variables d'env en Python](#3-env-python)
4. [Environnements virtuels Python](#4-venv)
5. [Requêtes HTTP — Python & Bash](#5-http)
6. [joblib — sauvegarder des objets ML](#6-joblib)
7. [GitLab — Repository & CI/CD](#7-gitlab)
8. [Docker — Dockerfile, volumes, docker-compose, DockerHub](#8-docker)
9. [Pytest — tests automatisés](#9-pytest)
10. [FastAPI + Pydantic + Prometheus](#10-fastapi)
11. [curl — requêtes HTTP depuis le terminal](#11-curl)
12. [Kubernetes — ressources essentielles](#12-kubernetes)
13. [Prometheus — configuration & PromQL](#13-prometheus)
14. [Grafana — datasources & dashboards](#14-grafana)


---
<a id="1-jupyter"></a>
## 1. Notebooks Jupyter

Un notebook est un document JSON (extension `.ipynb`) qui entremêle :
- **Cellules Markdown** — documentation, formules, titres
- **Cellules Code** — Python exécutable avec sortie inline
- **Cellules Raw** — texte brut (non exécuté)

### Raccourcis clavier essentiels (mode commande = touche Échap)
| Raccourci | Action |
|-----------|--------|
| `A` | Insérer cellule au-dessus |
| `B` | Insérer cellule en-dessous |
| `D D` | Supprimer la cellule |
| `M` | Passer en Markdown |
| `Y` | Passer en Code |
| `Shift+Enter` | Exécuter et passer à la suivante |
| `Ctrl+Enter` | Exécuter sans avancer |
| `Shift+M` | Fusionner les cellules sélectionnées |


In [ ]:
# 1.1 — Structure d'un fichier .ipynb (format JSON)
import json

# Un notebook est un simple fichier JSON
structure_ipynb = {
    "nbformat": 4,
    "nbformat_minor": 5,
    "metadata": {
        "kernelspec": {"name": "python3", "display_name": "Python 3"}
    },
    "cells": [
        {"cell_type": "markdown", "source": "# Mon titre"},
        {"cell_type": "code",     "source": "print('Hello')"},
    ]
}
print(json.dumps(obj=structure_ipynb, indent=2, ensure_ascii=False))

In [ ]:
# 1.2 — Lire un notebook depuis Python (utile pour les pipelines CI)

import json, pathlib

def lire_notebook(chemin: str) -> dict:
    with open(file=chemin, encoding="utf-8") as f:
        return json.load(fp=f)

def extraire_code(nb: dict) -> list[str]:
    """Retourne le code source de toutes les cellules code."""
    return [
        "".join(c["source"])
        for c in nb["cells"]
        if c["cell_type"] == "code"
    ]

def compter_cellules(nb: dict) -> dict:
    compteur = {}
    for c in nb["cells"]:
        t = c["cell_type"]
        compteur[t] = compteur.get(t, 0) + 1
    return compteur

# Exemple sur ce notebook lui-même
import sys
NB_PATH = __vsc_ipynb_file__ if hasattr(__builtins__, '__vsc_ipynb_file__') else None

# Simulation
nb_mock = {
    "cells": [
        {"cell_type": "markdown", "source": "# Titre"},
        {"cell_type": "code",     "source": "x = 1"},
        {"cell_type": "code",     "source": "print(x)"},
        {"cell_type": "raw",      "source": "texte brut"},
    ]
}
print("Compteur :", compter_cellules(nb=nb_mock))
print("Code extrait :", extraire_code(nb=nb_mock))

---
<a id="2-env-bash"></a>
## 2. Variables d'environnement — `export` & `.bashrc`

### `export` — déclarer une variable d'environnement dans le shell courant
```bash
export NOM_VARIABLE="valeur"       # déclare et exporte
export DB_HOST="localhost"
export DB_PORT=3306
export API_KEY="abc123"

echo $DB_HOST                      # afficher
printenv DB_HOST                   # aussi
env | grep DB_                     # filtrer

unset DB_HOST                      # supprimer
```

### `.bashrc` — persistance entre les sessions
```bash
# Ajouter dans ~/.bashrc :
export DB_HOST="localhost"
export DB_PORT=3306
export APP_ENV="development"
export PATH="$PATH:/opt/outils"

# Recharger sans fermer le terminal :
source ~/.bashrc
# ou :
. ~/.bashrc
```

### Portée des variables
```bash
MON_VAR="local"          # variable locale (non héritée par les sous-processus)
export MON_VAR="global"  # variable d'environnement (héritée par les sous-processus)

# Vérifier
bash -c 'echo $MON_VAR'  # vide si non exportée
```


In [ ]:
# 2.1 — Simulation bash en Python (pour comprendre les concepts)

import os, subprocess

# Équivalent de : export DB_HOST="localhost"
os.environ["DB_HOST"]   = "localhost"
os.environ["DB_PORT"]   = "3306"
os.environ["APP_ENV"]   = "development"
os.environ["API_KEY"]   = "secretkey123"

# Afficher toutes les variables définies (comme `env | grep DB_`)
db_vars = {k: v for k, v in os.environ.items() if k.startswith("DB_")}
print("Variables DB_ :", db_vars)

# Contenu type d'un fichier ~/.bashrc pour un projet data
bashrc_exemple = """
# === Projet TechShop ===
export DB_HOST="localhost"
export DB_PORT="3306"
export DB_NAME="techshop"
export DB_USER="app_user"
export DB_PASSWORD="motdepasse"

export ML_MODEL_PATH="$HOME/models/churn_v1.joblib"
export APP_ENV="development"
export PYTHONPATH="$PYTHONPATH:$HOME/techshop/src"

# Alias pratiques
alias py="python3"
alias k="kubectl"
alias dc="docker-compose"
"""
print("Contenu .bashrc exemple :")
print(bashrc_exemple)

---
<a id="3-env-python"></a>
## 3. Variables d'environnement en Python

| Méthode | Usage |
|---------|-------|
| `os.environ["CLE"]` | Lecture directe — `KeyError` si absente |
| `os.environ.get("CLE", "defaut")` | Avec valeur par défaut |
| `os.environ.get("CLE")` | Retourne `None` si absente |
| `load_dotenv(".env")` | Charger depuis un fichier `.env` |


In [ ]:
# 3.1 — Lecture des variables d'environnement
import os

# Valeur par défaut si absente
db_host = os.environ.get(key="DB_HOST", default="localhost")
db_port = int(os.environ.get(key="DB_PORT", default="3306"))
debug   = os.environ.get(key="DEBUG", default="false").lower() == "true"

print(f"DB : {db_host}:{db_port}")
print(f"Debug : {debug}")

# Variable obligatoire — lever une erreur explicite si absente
def require(key: str) -> str:
    val = os.environ.get(key=key)
    if not val:
        raise EnvironmentError(f"Variable d'environnement requise : {key}")
    return val

try:
    secret = require(key="SECRET_KEY_MANQUANTE")
except EnvironmentError as e:
    print(f"Erreur : {e}")

In [ ]:
# 3.2 — python-dotenv : charger un .env

# Créer un .env temporaire
env_content = """
DB_HOST=localhost
DB_PORT=3306
DB_NAME=techshop
ML_MODEL_PATH=./models/churn_v1.joblib
DEBUG=false
"""
with open(file="/tmp/.env_demo2", mode="w") as f:
    f.write(env_content.strip())

try:
    from dotenv import load_dotenv
    load_dotenv(dotenv_path="/tmp/.env_demo2", override=True)
    print("dotenv chargé")
except ImportError:
    for line in env_content.strip().splitlines():
        if "=" in line:
            k, v = line.split(sep="=", maxsplit=1)
            os.environ[k.strip()] = v.strip()
    print("(simulation dotenv)")

print(f"DB_HOST        = {os.environ.get(key='DB_HOST')}")
print(f"ML_MODEL_PATH  = {os.environ.get(key='ML_MODEL_PATH')}")
print(f"DEBUG          = {os.environ.get(key='DEBUG')}")

# Bonne pratique : encapsuler dans un objet config
from dataclasses import dataclass

@dataclass
class Config:
    db_host:    str   = os.environ.get(key="DB_HOST",  default="localhost")
    db_port:    int   = int(os.environ.get(key="DB_PORT", default="3306"))
    db_name:    str   = os.environ.get(key="DB_NAME",  default="techshop")
    model_path: str   = os.environ.get(key="ML_MODEL_PATH", default="./model.joblib")
    debug:      bool  = os.environ.get(key="DEBUG", default="false").lower() == "true"

cfg = Config()
print(f"\nConfig : {cfg}")

---
<a id="4-venv"></a>
## 4. Environnements virtuels Python

Un venv isole les dépendances de chaque projet (évite les conflits de versions).

```bash
# ── Créer un venv ──────────────────────────────────────────────────
python3 -m venv .venv                   # crée .venv/ dans le dossier courant
python3 -m venv .venv --python=3.11     # forcer une version Python

# ── Activer ────────────────────────────────────────────────────────
source .venv/bin/activate               # Linux/Mac
.venv\Scripts\activate               # Windows

# Prompt change : (.venv) $ ...

# ── Installer des paquets ──────────────────────────────────────────
pip install fastapi uvicorn pandas scikit-learn
pip install -r requirements.txt
pip install -e .                        # installer le projet en mode éditable

# ── Sauvegarder les dépendances ────────────────────────────────────
pip freeze > requirements.txt
# ou avec pip-tools (recommandé) :
pip-compile requirements.in             # génère requirements.txt versionné

# ── Désactiver ─────────────────────────────────────────────────────
deactivate

# ── Supprimer ──────────────────────────────────────────────────────
rm -rf .venv
```

### Structure recommandée d'un projet Python
```
mon_projet/
├── .venv/              # jamais committer (.gitignore)
├── src/
│   └── techshop/
│       ├── __init__.py
│       ├── api.py
│       └── models.py
├── tests/
│   └── test_api.py
├── requirements.txt
├── requirements-dev.txt   # pytest, flake8, black...
├── .env                   # jamais committer
├── .gitignore
└── pyproject.toml
```


In [ ]:
# 4.1 — requirements.txt et pyproject.toml

requirements_txt = """
# requirements.txt
fastapi==0.111.0
uvicorn[standard]==0.29.0
pydantic==2.7.1
pandas==2.2.2
scikit-learn==1.4.2
joblib==1.4.2
sqlalchemy==2.0.30
pymysql==1.1.1
python-dotenv==1.0.1
prometheus-fastapi-instrumentator==6.1.0
httpx==0.27.0
"""

requirements_dev = """
# requirements-dev.txt (dépendances de développement)
pytest==8.2.0
pytest-asyncio==0.23.6
httpx==0.27.0
flake8==7.0.0
black==24.4.2
mypy==1.10.0
"""

pyproject_toml = """
# pyproject.toml
[project]
name = "techshop"
version = "1.0.0"
requires-python = ">=3.10"
dependencies = [
    "fastapi>=0.111",
    "pydantic>=2.0",
    "scikit-learn>=1.4",
]

[project.optional-dependencies]
dev = ["pytest", "httpx", "flake8"]

[tool.pytest.ini_options]
testpaths = ["tests"]
asyncio_mode = "auto"

[tool.black]
line-length = 100
"""

print("requirements.txt :", requirements_txt)
print("requirements-dev.txt :", requirements_dev)
print("pyproject.toml :", pyproject_toml)

---
<a id="5-http"></a>
## 5. Requêtes HTTP — Python & Bash

| Outil | Usage |
|-------|-------|
| `requests` | Librairie Python classique, synchrone |
| `httpx` | Moderne, supporte async |
| `urllib.request` | Stdlib Python, sans dépendance |
| `curl` | Ligne de commande Bash |


In [ ]:
# 5.1 — requests (synchrone)
import requests, json

BASE_URL = "https://httpbin.org"   # API de test publique

# GET
response = requests.get(url=f"{BASE_URL}/get", params={"user": "alice", "page": 1})
print(f"GET — Status : {response.status_code}")
print(f"URL réelle  : {response.url}")
data = response.json()
print(f"Args reçus  : {data['args']}")

# POST avec JSON
payload = {"nom": "Alice Martin", "commande_id": 101, "montant": 299.99}
response = requests.post(url=f"{BASE_URL}/post",
                         json=payload,
                         headers={"Authorization": "Bearer mon_token"})
print(f"\nPOST — Status : {response.status_code}")
data = response.json()
print(f"JSON envoyé : {data['json']}")

# Gestion des erreurs
try:
    r = requests.get(url=f"{BASE_URL}/status/404", timeout=5)
    r.raise_for_status()              # lève HTTPError si 4xx/5xx
except requests.exceptions.HTTPError as e:
    print(f"\nErreur HTTP : {e}")
except requests.exceptions.ConnectionError:
    print("Erreur de connexion")
except requests.exceptions.Timeout:
    print("Timeout")

In [ ]:
# 5.2 — httpx (sync + async)
try:
    import httpx

    # Synchrone
    with httpx.Client(base_url="https://httpbin.org", timeout=10.0) as client:
        r = client.get(url="/get", params={"source": "httpx"})
        print(f"httpx GET — Status : {r.status_code}")
        print(f"Args : {r.json()['args']}")

    print("\n--- Code async (pour FastAPI tests, etc.) ---")
    async_example = '''
import httpx, asyncio

async def fetch(url: str):
    async with httpx.AsyncClient() as client:
        response = await client.get(url=url)
        return response.json()

# Dans un contexte async :
# result = await fetch("https://api.example.com/data")
# Dans un script :
# result = asyncio.run(fetch("https://api.example.com/data"))
    '''
    print(async_example)

except ImportError:
    print("httpx non installé — pip install httpx")

In [ ]:
# 5.3 — urllib (stdlib, pas de dépendance)
from urllib.request import urlopen, Request
from urllib.parse   import urlencode
import urllib.error

url = "https://httpbin.org/get?" + urlencode(query={"q": "techshop", "page": "1"})
req = Request(url=url, headers={"User-Agent": "TechShop/1.0"})

try:
    with urlopen(url=req, timeout=10) as resp:
        body = json.loads(s=resp.read())
        print(f"Status : {resp.status}")
        print(f"Args   : {body['args']}")
except urllib.error.HTTPError as e:
    print(f"Erreur HTTP {e.code}: {e.reason}")
except urllib.error.URLError as e:
    print(f"Erreur URL : {e.reason}")

In [ ]:
# 5.4 — Simulation de curl depuis Python (subprocess)
import subprocess

# Équivalent bash de : curl -s https://httpbin.org/get
def curl_get(url: str, headers: dict = None) -> str:
    cmd = ["curl", "-s", url]
    if headers:
        for k, v in headers.items():
            cmd += ["-H", f"{k}: {v}"]
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
    return result.stdout

# Simulation (sans exécution pour ne pas dépendre de curl installé)
print("Exemples de commandes curl :")
print("  curl -s https://httpbin.org/get")
print("  curl -s -X POST -H 'Content-Type: application/json'")
print("       -d '{\"key\":\"value\"}' https://httpbin.org/post")
print("  curl -s -o /dev/null -w '%{http_code}' https://httpbin.org/status/200")

---
<a id="6-joblib"></a>
## 6. joblib — sauvegarder des objets ML

`joblib` sérialise efficacement les objets Python volumineux (matrices NumPy, modèles sklearn).

| Fonction | Usage |
|----------|-------|
| `joblib.dump(obj, path)` | Sauvegarder |
| `joblib.load(path)` | Charger |
| `joblib.dump(obj, path, compress=3)` | Avec compression (0-9) |
| `joblib.Parallel(n_jobs=-1)` | Calcul parallèle |


In [ ]:
# 6.1 — Sauvegarder et charger un pipeline ML complet
import joblib, numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
import datetime

# Créer un dataset mock
X, y = make_classification(n_samples=500, n_features=8, random_state=42)

# Entraîner un pipeline
pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("clf",    RandomForestClassifier(n_estimators=50, random_state=42))
])
pipe.fit(X=X, y=y)
print(f"Pipeline entraîné — Accuracy : {pipe.score(X=X, y=y):.3f}")

# ── Sauvegarde simple ──────────────────────────────────────────
model_path = Path("/tmp/techshop_churn.joblib")
joblib.dump(value=pipe, filename=model_path)
print(f"\nSauvegardé  : {model_path} ({model_path.stat().st_size/1024:.1f} Ko)")

# ── Avec compression ──────────────────────────────────────────
model_gz = Path("/tmp/techshop_churn.joblib.gz")
joblib.dump(value=pipe, filename=model_gz, compress=("gzip", 3))
print(f"Compressé   : {model_gz} ({model_gz.stat().st_size/1024:.1f} Ko)")

# ── Chargement ────────────────────────────────────────────────
pipe_reload = joblib.load(filename=model_path)
preds = pipe_reload.predict(X=X[:5])
print(f"\nPrédictions après rechargement : {preds}")

# ── Bundle avec métadonnées ───────────────────────────────────
bundle = {
    "model":        pipe,
    "feature_names":["age","anciennete","nb_cmds","ca","f5","f6","f7","f8"],
    "target":       "churn",
    "accuracy":     pipe.score(X=X, y=y),
    "date":         datetime.date.today().isoformat(),
    "version":      "2.0",
    "auteur":       "equipe-ml",
}
joblib.dump(value=bundle, filename="/tmp/churn_bundle.joblib")
b = joblib.load(filename="/tmp/churn_bundle.joblib")
print(f"\nBundle rechargé — version {b['version']}, accuracy {b['accuracy']:.3f}")

In [ ]:
# 6.2 — joblib.Parallel : calcul parallèle
from joblib import Parallel, delayed
import time

def traiter_batch(batch_id: int, taille: int) -> dict:
    """Simule le traitement d'un batch."""
    # time.sleep(0.1)  # simuler du travail
    return {"batch_id": batch_id, "nb_traite": taille, "statut": "ok"}

# Séquentiel
import time
t0 = time.time()
resultats_seq = [traiter_batch(batch_id=i, taille=100) for i in range(20)]
print(f"Séquentiel  : {time.time()-t0:.3f}s")

# Parallèle (n_jobs=-1 = tous les CPU)
t0 = time.time()
resultats_par = Parallel(n_jobs=-1)(
    delayed(function=traiter_batch)(batch_id=i, taille=100) for i in range(20)
)
print(f"Parallèle   : {time.time()-t0:.3f}s")
print(f"Résultats   : {len(resultats_par)} batches traités")

# Autre usage : scorer plusieurs modèles en parallèle
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

modeles = [
    ("LR",  LogisticRegression(max_iter=500)),
    ("DT",  DecisionTreeClassifier()),
    ("RF",  RandomForestClassifier(n_estimators=50)),
]

scores = Parallel(n_jobs=-1)(
    delayed(function=cross_val_score)(estimator=m, X=X, y=y, cv=3, scoring="accuracy")
    for _, m in modeles
)
for (nom, _), score in zip(modeles, scores):
    print(f"  {nom}: {score.mean():.3f} ± {score.std():.3f}")

---
<a id="7-gitlab"></a>
## 7. GitLab — Repository, CI/CD, Runner

### Workflow standard
```
git add . → git commit → git push → Pipeline CI/CD déclenché
```


In [ ]:
# 7.1 — .gitlab-ci.yml : anatomie et exemples

gitlab_ci_yaml = """
# ════════════════════════════════════════════════════════════
# .gitlab-ci.yml — Pipeline pour projet FastAPI + ML
# ════════════════════════════════════════════════════════════

image: python:3.11-slim

variables:
  PIP_CACHE_DIR: "$CI_PROJECT_DIR/.pip-cache"
  APP_ENV: "ci"
  # Variables secrètes à définir dans Settings > CI/CD > Variables :
  # DB_PASSWORD, API_KEY, REGISTRY_TOKEN...

cache:
  key: "$CI_COMMIT_REF_SLUG"
  paths:
    - .pip-cache/
    - .venv/

stages:
  - install
  - test
  - build
  - deploy

# ── Stage install ─────────────────────────────────────────
setup:
  stage: install
  script:
    - python -m venv .venv
    - source .venv/bin/activate
    - pip install --upgrade pip
    - pip install -r requirements.txt -r requirements-dev.txt
  artifacts:
    paths: [.venv/]
    expire_in: 2 hours

# ── Stage test ────────────────────────────────────────────
pytest:
  stage: test
  needs: [setup]
  script:
    - source .venv/bin/activate
    - pytest tests/ -v --tb=short
          --junitxml=junit.xml
          --cov=src --cov-report=xml
  coverage: '/TOTAL.*\s+(\d+%)$/'   # extrait le % de couverture
  artifacts:
    reports:
      junit:   junit.xml
      coverage_report:
        coverage_format: cobertura
        path: coverage.xml

# ── Stage build image Docker ──────────────────────────────
build:
  stage: build
  image: docker:24
  services: [docker:24-dind]
  variables:
    DOCKER_TLS_CERTDIR: "/certs"
  script:
    - docker login -u $CI_REGISTRY_USER -p $CI_REGISTRY_PASSWORD $CI_REGISTRY
    - docker build -t $CI_REGISTRY_IMAGE:$CI_COMMIT_SHA .
    - docker build -t $CI_REGISTRY_IMAGE:latest .
    - docker push $CI_REGISTRY_IMAGE:$CI_COMMIT_SHA
    - docker push $CI_REGISTRY_IMAGE:latest
  only: [main, tags]

# ── Stage deploy ──────────────────────────────────────────
deploy_k8s:
  stage: deploy
  image: bitnami/kubectl:latest
  script:
    - kubectl set image deployment/techshop-api api=$CI_REGISTRY_IMAGE:$CI_COMMIT_SHA
             -n techshop-prod
    - kubectl rollout status deployment/techshop-api -n techshop-prod
  environment:
    name: production
    url: https://api.techshop.fr
  when: manual
  only: [main]
"""
print(gitlab_ci_yaml)

In [ ]:
# 7.2 — Commandes gitlab-runner

print("""
=== Installation du Runner ===
curl -L https://packages.gitlab.com/install/repositories/runner/gitlab-runner/script.deb.sh | sudo bash
sudo apt-get install -y gitlab-runner

=== Enregistrement ===
sudo gitlab-runner register \\
    --url "http://gitlab.example.com" \\
    --registration-token "TOKEN" \\
    --description "Runner Docker" \\
    --executor "docker" \\
    --docker-image "python:3.11-slim"

=== Gestion du service ===
sudo gitlab-runner start      # démarrer
sudo gitlab-runner stop       # arrêter
sudo gitlab-runner restart    # redémarrer
sudo gitlab-runner status     # état

=== Debugging ===
sudo gitlab-runner list              # runners enregistrés
sudo gitlab-runner verify            # tester la connexion GitLab
sudo gitlab-runner exec docker pytest  # lancer le job 'pytest' localement
""")

---
<a id="8-docker"></a>
## 8. Docker — Dockerfile, volumes, docker-compose, DockerHub


In [ ]:
# 8.1 — Dockerfile pour une API FastAPI

dockerfile = """
# ════════════════════════════════════════════════════════════
# Dockerfile — Application FastAPI TechShop
# ════════════════════════════════════════════════════════════

# Image de base légère
FROM python:3.11-slim

# Métadonnées
LABEL maintainer="equipe@techshop.fr"
LABEL version="1.2.0"

# Répertoire de travail dans le container
WORKDIR /app

# Copier les dépendances en premier (optimise le cache Docker)
COPY requirements.txt .
RUN pip install --no-cache-dir --upgrade pip \\
 && pip install --no-cache-dir -r requirements.txt

# Copier le code source
COPY src/ ./src/
COPY models/ ./models/

# Variables d'environnement par défaut (surchargeables au runtime)
ENV APP_ENV=production
ENV PORT=8000
ENV LOG_LEVEL=INFO

# Exposer le port (documentation uniquement)
EXPOSE 8000

# Créer un utilisateur non-root (sécurité)
RUN useradd -m -u 1000 appuser
USER appuser

# Point d'entrée : démarrer uvicorn
CMD ["uvicorn", "src.main:app", "--host", "0.0.0.0", "--port", "8000"]

# Health check intégré
HEALTHCHECK --interval=30s --timeout=10s --start-period=5s --retries=3 \\
    CMD curl -f http://localhost:8000/health || exit 1
"""

print("=== Dockerfile ===")
print(dockerfile)

print("""
=== Commandes Docker ===
# Build
docker build -t techshop-api:latest .
docker build -t techshop-api:v1.2 --build-arg ENV=prod .

# Run
docker run -p 8000:8000 techshop-api:latest
docker run -d --name api -p 8000:8000 -e DB_HOST=localhost techshop-api:latest
docker run -d -v $(pwd)/data:/app/data techshop-api:latest

# Inspecter
docker ps                          # containers actifs
docker logs api -f                 # logs en temps réel
docker exec -it api bash           # shell dans le container
docker inspect api                 # JSON détaillé

# Images
docker images
docker rmi techshop-api:latest
docker image prune                 # nettoyer les images inutilisées
""")

In [ ]:
# 8.2 — Volumes Docker

print("""
=== Types de volumes ===

# ── Volume nommé (géré par Docker, recommandé pour les BDD) ──
docker volume create techshop_db_data
docker run -v techshop_db_data:/var/lib/mysql mysql:8.0
docker volume ls
docker volume inspect techshop_db_data
docker volume rm techshop_db_data

# ── Bind mount (chemin local → container) ────────────────────
docker run -v /home/user/data:/app/data techshop-api        # absolu
docker run -v $(pwd)/models:/app/models techshop-api         # relatif (Linux)

# ── tmpfs mount (RAM, temporaire, non persistant) ─────────────
docker run --tmpfs /tmp:rw,size=100m techshop-api

=== Dans docker-compose ===
services:
  db:
    volumes:
      - db_data:/var/lib/mysql          # volume nommé
      - ./sql/init.sql:/docker-entrypoint-initdb.d/init.sql  # bind mount

volumes:
  db_data:                              # déclaration volume nommé
""")

In [ ]:
# 8.3 — docker-compose.yml complet avec depends_on

compose_yaml = """
version: "3.9"

services:

  # ── API FastAPI ────────────────────────────────────────────
  api:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: techshop_api
    restart: unless-stopped
    env_file: .env
    ports:
      - "8000:8000"
    volumes:
      - ./models:/app/models        # modèles ML
      - ./logs:/app/logs
    depends_on:
      db:
        condition: service_healthy  # attend que MySQL soit prêt
    networks: [techshop_net]
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3

  # ── Base de données MySQL ──────────────────────────────────
  db:
    image: mysql:8.0
    container_name: techshop_db
    restart: always
    env_file: .env
    environment:
      MYSQL_ROOT_PASSWORD: ${DB_ROOT_PASSWORD}
      MYSQL_DATABASE:      ${DB_NAME}
      MYSQL_USER:          ${DB_USER}
      MYSQL_PASSWORD:      ${DB_PASSWORD}
    volumes:
      - db_data:/var/lib/mysql
      - ./config/mysql/init:/docker-entrypoint-initdb.d:ro   # crée le compte exporter (voir 8.3 bis)
    networks: [techshop_net]
    healthcheck:
      test: ["CMD","mysqladmin","ping","-h","localhost"]
      interval: 10s
      timeout: 5s
      retries: 5

  # ── Prometheus (monitoring) ────────────────────────────────
  prometheus:
    image: prom/prometheus:latest
    container_name: techshop_prometheus
    command:
      - --config.file=/etc/prometheus/prometheus.yml
      - --storage.tsdb.path=/prometheus
      - --web.enable-lifecycle        # autorise POST /-/reload (recharger sans redémarrer)
    volumes:
      - ./config/prometheus.yml:/etc/prometheus/prometheus.yml:ro
      - ./config/rules:/etc/prometheus/rules:ro   # recording_rules.yml + alert_rules.yml (13.4, 13.5)
      - prometheus_data:/prometheus
    ports:
      - "9090:9090"
    depends_on:
      api:
        condition: service_healthy
      mysql-exporter:
        condition: service_started
      node-exporter:
        condition: service_started
      alertmanager:
        condition: service_started
    networks: [techshop_net]

  # ── Alertmanager (routage des alertes : email, Slack…) ──────
  alertmanager:
    image: prom/alertmanager:v0.27.0
    container_name: techshop_alertmanager
    restart: unless-stopped
    command:
      - --config.file=/etc/alertmanager/alertmanager.yml
      - --storage.path=/alertmanager  # silences et état des notifications
    volumes:
      - ./config/alertmanager:/etc/alertmanager:ro   # alertmanager.yml + secrets/ (13.6)
      - alertmanager_data:/alertmanager
    ports:
      - "9093:9093"
    networks: [techshop_net]

  # ── Pushgateway (métriques poussées par les jobs batch) ────
  pushgateway:
    image: prom/pushgateway:v1.9.0
    container_name: techshop_pushgateway
    restart: unless-stopped
    ports:
      - "9091:9091"
    networks: [techshop_net]

  # ── mysqld_exporter (métriques MySQL → format Prometheus) ──
  mysql-exporter:                   # nom = cible "mysql-exporter:9104" du prometheus.yml
    image: prom/mysqld-exporter:v0.15.1
    container_name: techshop_mysql_exporter
    restart: unless-stopped
    command:
      - --mysqld.address=db:3306    # service "db" sur le réseau techshop_net
      - --mysqld.username=exporter  # compte créé par le script d'init MySQL
    environment:
      MYSQLD_EXPORTER_PASSWORD: ${EXPORTER_PASSWORD}   # mot de passe lu depuis .env
    ports:
      - "9104:9104"                 # optionnel : curl localhost:9104/metrics
    depends_on:
      db:
        condition: service_healthy
    networks: [techshop_net]

  # ── node_exporter (métriques machine : CPU, RAM, disque) ───
  node-exporter:                    # nom = cible "node-exporter:9100"
    image: prom/node-exporter:v1.8.2
    container_name: techshop_node_exporter
    restart: unless-stopped
    command:
      - --path.procfs=/host/proc
      - --path.sysfs=/host/sys
      - --path.rootfs=/rootfs
    volumes:
      - /proc:/host/proc:ro
      - /sys:/host/sys:ro
      - /:/rootfs:ro
    networks: [techshop_net]

  # ── Grafana (dashboards) ───────────────────────────────────
  grafana:
    image: grafana/grafana:latest
    container_name: techshop_grafana
    ports:
      - "3000:3000"
    volumes:
      - grafana_data:/var/lib/grafana
      - ./config/grafana/datasources:/etc/grafana/provisioning/datasources
    depends_on: [prometheus]
    networks: [techshop_net]
    environment:
      GF_SECURITY_ADMIN_PASSWORD: admin

networks:
  techshop_net:
    driver: bridge

volumes:
  db_data:
  prometheus_data:
  alertmanager_data:
  grafana_data:
"""
print(compose_yaml)

In [ ]:
# 8.3 bis — Compte MySQL pour mysqld_exporter + fichier .env

env_file = """
# .env  (à la racine du projet, à côté du docker-compose.yml — NE PAS committer)
DB_ROOT_PASSWORD=root_pwd_a_changer
DB_NAME=techshop
DB_USER=techshop
DB_PASSWORD=techshop_pwd_a_changer
# Lu par db (script d'init) ET par mysql-exporter :
EXPORTER_PASSWORD=exporter_pwd_a_changer
"""

init_script = """
#!/bin/bash
# config/mysql/init/02-exporter.sh
# Exécuté automatiquement par l'image mysql au PREMIER démarrage
# (uniquement si le volume db_data est vide).
# .sh plutôt que .sql : un .sql ne peut pas lire les variables d'environnement.
set -e
mysql -uroot -p"${MYSQL_ROOT_PASSWORD}" <<SQL
CREATE USER IF NOT EXISTS 'exporter'@'%'
  IDENTIFIED BY '${EXPORTER_PASSWORD}'
  WITH MAX_USER_CONNECTIONS 3;
-- droits minimum en LECTURE pour collecter les métriques
GRANT PROCESS, REPLICATION CLIENT, SELECT ON *.* TO 'exporter'@'%';
FLUSH PRIVILEGES;
SQL
"""

print(".env :")
print(env_file)
print("config/mysql/init/02-exporter.sh :")
print(init_script)

print("""
=== Arborescence ===
projet/
├── docker-compose.yml
├── .env
└── config/
    ├── prometheus.yml                 → 13.2
    ├── rules/
    │   ├── recording_rules.yml        → 13.4
    │   ├── alert_rules.yml            → 13.5
    │   └── tests/rules_test.yml       → 13.7
    ├── alertmanager/
    │   ├── alertmanager.yml           → 13.6
    │   └── secrets/                   → smtp_password, slack_webhook_url (dans .gitignore !)
    ├── mysql/init/02-exporter.sh      → monté dans /docker-entrypoint-initdb.d
    └── grafana/datasources/prometheus.yml

=== Comment ça s'enchaîne ===
1. db démarre → l'image mysql exécute /docker-entrypoint-initdb.d/*.sh
   → le compte 'exporter' est créé (env_file: .env fournit EXPORTER_PASSWORD)
2. db devient healthy → mysql-exporter démarre et se connecte à db:3306
3. mysql-exporter expose http://mysql-exporter:9104/metrics
4. Prometheus scrape cette URL toutes les 15s (job 'mysql' du prometheus.yml)

=== Vérifications ===
docker compose up -d
docker compose logs mysql-exporter          # pas d'erreur "Access denied"
curl -s localhost:9104/metrics | grep mysql_up   # mysql_up 1 = connexion OK
# Prometheus → http://localhost:9090/targets : jobs mysql et node en UP

# Volume déjà créé avant l'ajout du script ? L'init ne rejoue PAS :
docker compose exec db mysql -uroot -p -e "CREATE USER 'exporter'@'%' IDENTIFIED BY '...'; \\
  GRANT PROCESS, REPLICATION CLIENT, SELECT ON *.* TO 'exporter'@'%';"
# (ou repartir de zéro : docker compose down -v  → SUPPRIME les données)
""")

In [ ]:
# 8.4 — DockerHub : push/pull d'images

print("""
=== DockerHub — Publier une image ===

# 1. Se connecter
docker login
# ou avec token :
echo $DOCKERHUB_TOKEN | docker login -u moncompte --password-stdin

# 2. Tagger l'image avec votre compte DockerHub
docker build -t moncompte/techshop-api:v1.2 .
docker tag techshop-api:latest moncompte/techshop-api:latest

# 3. Pousser
docker push moncompte/techshop-api:v1.2
docker push moncompte/techshop-api:latest

# 4. Tirer depuis n'importe où
docker pull moncompte/techshop-api:v1.2
docker pull moncompte/techshop-api:latest

=== Bonnes pratiques tags ===
moncompte/techshop-api:latest         # dernière version stable
moncompte/techshop-api:v1.2.0         # version sémantique
moncompte/techshop-api:main-abc123    # branche + SHA court (CI)
moncompte/techshop-api:dev            # branche de dev

=== Dans le CI .gitlab-ci.yml ===
script:
  - docker login -u $DOCKERHUB_USER -p $DOCKERHUB_TOKEN
  - docker push $DOCKERHUB_USER/techshop-api:$CI_COMMIT_SHORT_SHA
""")

---
<a id="9-pytest"></a>
## 9. Pytest — framework de tests

### Conventions
- Fichiers : `test_*.py` ou `*_test.py`
- Fonctions : `def test_*()`
- Classes : `class Test*`
- `assert` natif Python — pas besoin de `.assertEqual()`

### Commandes
```bash
pytest                              # tous les tests
pytest tests/test_api.py            # fichier spécifique
pytest -v                           # verbose
pytest -k "churn"                   # filtrer par nom
pytest -x                           # arrêter au premier échec
pytest --tb=short                   # traceback court
pytest --cov=src --cov-report=html  # couverture de code
```


In [ ]:
# 9.1 — Tests unitaires avec pytest (exécutables directement)
import pytest

# ── Code à tester ────────────────────────────────────────────────────
def calculer_ttc(ht: float, tva: float = 0.20) -> float:
    if ht < 0:
        raise ValueError("Le montant HT ne peut pas être négatif")
    return round(number=ht * (1 + tva), ndigits=2)

def segmenter_client(ca: float) -> str:
    if ca < 100:   return "Bronze"
    if ca < 500:   return "Silver"
    if ca < 2000:  return "Gold"
    return "Platinum"

# ── Tests ────────────────────────────────────────────────────────────

def test_calculer_ttc_standard():
    assert calculer_ttc(ht=100.0) == 120.0

def test_calculer_ttc_sans_tva():
    assert calculer_ttc(ht=100.0, tva=0) == 100.0

def test_calculer_ttc_tva_reduite():
    assert calculer_ttc(ht=100.0, tva=0.055) == 105.5

def test_calculer_ttc_negatif():
    with pytest.raises(expected_exception=ValueError, match="négatif"):
        calculer_ttc(ht=-10.0)

def test_segmenter_bronze():
    assert segmenter_client(ca=50)   == "Bronze"
    assert segmenter_client(ca=0)    == "Bronze"

def test_segmenter_silver():
    assert segmenter_client(ca=100)  == "Silver"
    assert segmenter_client(ca=499)  == "Silver"

def test_segmenter_platinum():
    assert segmenter_client(ca=5000) == "Platinum"

# Lancer directement depuis le notebook
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pytest", __file__, "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout[-3000:] if len(result.stdout) > 3000 else result.stdout)

In [2]:
# 9.2 — Fixtures, parametrize, monkeypatch

# ── Fixtures ──────────────────────────────────────────────────────────
import pytest, os

@pytest.fixture
def client_gold():
    """Fixture : client Gold réutilisable."""
    return {"client_id": 1, "nom": "Alice", "segment": "Gold", "ca_total": 1500.0}

@pytest.fixture
def db_vide():
    """Fixture : base de données temporaire."""
    clients = []
    yield clients
    # teardown : nettoyage après le test
    clients.clear()

def test_client_gold_actif(client_gold):
    assert client_gold["segment"] == "Gold"
    assert client_gold["ca_total"] >= 500

def test_ajouter_client(db_vide):
    db_vide.append({"id": 1, "nom": "Bob"})
    assert len(db_vide) == 1

# ── parametrize : tester plusieurs valeurs ────────────────────────────
@pytest.mark.parametrize(argnames="ht,tva,attendu", argvalues=[
    (100.0, 0.20, 120.0),
    (50.0,  0.10, 55.0),
    (200.0, 0.055, 211.0),
    (0.0,   0.20, 0.0),
])
def test_calculer_ttc_parametrise(ht, tva, attendu):
    assert calculer_ttc(ht=ht, tva=tva) == attendu

# ── monkeypatch : patcher les variables d'env ─────────────────────────
def get_api_url():
    return os.environ.get(key="API_URL", default="http://localhost:8000")

def test_api_url_defaut(monkeypatch):
    monkeypatch.delenv(name="API_URL", raising=False)
    assert get_api_url() == "http://localhost:8000"

def test_api_url_env(monkeypatch):
    monkeypatch.setenv(name="API_URL", value="http://prod.api.fr")
    assert get_api_url() == "http://prod.api.fr"

print("Exemples de tests écrits — lancer avec : pytest tests/ -v")
print("Fixtures définies : client_gold, db_vide")
print("Parametrize : 4 cas de test pour calculer_ttc")

Exemples de tests écrits — lancer avec : pytest tests/ -v
Fixtures définies : client_gold, db_vide
Parametrize : 4 cas de test pour calculer_ttc


---
<a id="10-fastapi"></a>
## 10. FastAPI + Pydantic + prometheus-fastapi-instrumentator

### Architecture d'une API ML
```
Client → [POST /predict] → FastAPI → joblib.load(model) → Prédiction JSON
                         ↓
                    [GET /metrics] ← Prometheus scrape toutes les 15s
```


In [ ]:
# 10.1 — Modèles Pydantic (validation des données)

from pydantic import BaseModel, Field, field_validator
from typing import Optional, Literal
import datetime

# ── Modèle de requête (données entrantes) ────────────────────────────
class ClientFeatures(BaseModel):
    """Features nécessaires pour prédire le churn."""
    age:             float = Field(default=..., gt=0, lt=120, description="Âge en années")
    anciennete_mois: int   = Field(default=..., ge=0, description="Ancienneté en mois")
    nb_commandes:    int   = Field(default=..., ge=0)
    ca_total:        float = Field(default=..., ge=0)
    ville:           str
    segment:         Literal["Bronze","Silver","Gold","Platinum"]

    @field_validator("age")
    @classmethod
    def age_coherent(cls, v: float) -> float:
        if v > 110:
            raise ValueError("Âge improbable (> 110 ans)")
        return v

# ── Modèle de réponse (données sortantes) ────────────────────────────
class PredictionResponse(BaseModel):
    client_id:     Optional[int]
    churn_proba:   float = Field(default=..., ge=0, le=1)
    churn_predict: int   = Field(default=..., ge=0, le=1)
    risque:        Literal["Faible","Moyen","Élevé"]
    modele_version: str
    horodatage:    datetime.datetime = Field(default_factory=datetime.datetime.now)

# Validation automatique
try:
    f = ClientFeatures(age=28, anciennete_mois=6, nb_commandes=3,
                       ca_total=450.0, ville="Paris", segment="Silver")
    print("Valide :", f.model_dump())
except Exception as e:
    print("Erreur :", e)

# Test validation
try:
    ClientFeatures(age=150, anciennete_mois=3, nb_commandes=1,
                   ca_total=100.0, ville="Lyon", segment="Bronze")
except Exception as e:
    print(f"\nValidation age : {e}")

try:
    ClientFeatures(age=30, anciennete_mois=3, nb_commandes=1,
                   ca_total=100.0, ville="Lyon", segment="Ultra")
except Exception as e:
    print(f"Validation segment : type invalide détecté")

In [ ]:
# 10.2 — Application FastAPI complète

fastapi_app_code = '''
# main.py — Application FastAPI TechShop

import os, joblib, numpy as np
from fastapi import FastAPI, HTTPException
from prometheus_fastapi_instrumentator import Instrumentator
from pydantic import BaseModel, Field
from typing import Literal
import datetime

app = FastAPI(
    title="TechShop Churn API",
    description="Prédiction du churn client",
    version="1.0.0"
)

# ── Instrumentation Prometheus ────────────────────────────────────────
# Expose automatiquement /metrics avec :
#   - http_requests_total (compteur par méthode/endpoint/status)
#   - http_request_duration_seconds (histogramme)
#   - http_requests_in_progress (gauge)
instrumentator = Instrumentator(
    should_group_status_codes=True,
    should_ignore_untemplated=True,
    should_respect_env_var=True,
    should_instrument_requests_inprogress=True,
)
instrumentator.instrument(app=app).expose(app=app)    # ajoute GET /metrics

# ── Chargement du modèle (au démarrage) ──────────────────────────────
model = None

@app.on_event(event_type="startup")
async def load_model():
    global model
    path = os.environ.get(key="ML_MODEL_PATH", default="./models/churn.joblib")
    model = joblib.load(filename=path)
    print(f"Modèle chargé depuis {path}")

# ── Endpoints ─────────────────────────────────────────────────────────
@app.get(path="/health")
def health():
    return {"status": "ok", "model_loaded": model is not None}

@app.get(path="/ready")
def ready():
    if model is None:
        raise HTTPException(status_code=503, detail="Modèle non chargé")
    return {"status": "ready"}

@app.post(path="/predict", response_model=PredictionResponse)
def predict(features: ClientFeatures):
    if model is None:
        raise HTTPException(status_code=503, detail="Modèle non disponible")

    X = np.array(object=[[
        features.age, features.anciennete_mois, features.nb_commandes,
        features.ca_total
    ]])
    proba = model.predict_proba(X=X)[0, 1]
    pred  = int(proba >= 0.5)

    risque = "Faible" if proba < 0.3 else ("Moyen" if proba < 0.6 else "Élevé")

    return PredictionResponse(
        client_id=None,
        churn_proba=round(number=proba, ndigits=4),
        churn_predict=pred,
        risque=risque,
        modele_version="1.0.0"
    )

@app.get(path="/metrics/custom")
def custom_metrics():
    """Métriques métier personnalisées."""
    return {
        "nb_predictions_total": 1234,
        "taux_churn_predit":    0.23,
        "latence_moyenne_ms":   45.2
    }
'''
print(fastapi_app_code)
print("\nDémarrer avec : uvicorn main:app --host 0.0.0.0 --port 8000 --reload")

---
<a id="11-curl"></a>
## 11. curl — requêtes HTTP depuis le terminal

```bash
# Options essentielles :
# -s           : silencieux (pas de barre de progression)
# -X METHOD    : méthode HTTP (GET par défaut)
# -H "..."     : ajouter un header
# -d "..."     : corps de la requête (active POST)
# --json "..." : corps JSON + Content-Type auto (curl >= 7.82)
# -o fichier   : sauvegarder dans un fichier
# -w "..."     : afficher des infos après la requête
# -v           : verbose (headers de requête et réponse)
# -L           : suivre les redirections
# -u user:pwd  : authentification Basic
```


In [ ]:
# 11.1 — Exemples curl complets

print("""
=== GET simple ===
curl -s https://httpbin.org/get | python3 -m json.tool

=== GET avec paramètres ===
curl -s "https://httpbin.org/get?user=alice&page=1"

=== POST JSON ===
curl -s -X POST \\
     -H "Content-Type: application/json" \\
     -d '{"nom": "Alice", "ca": 450.0, "segment": "Silver"}' \\
     http://localhost:8000/predict

=== POST avec --json (curl >= 7.82, plus simple) ===
curl -s --json '{"age":28,"anciennete_mois":6,"nb_commandes":3,"ca_total":450,"ville":"Paris","segment":"Silver"}' \\
     http://localhost:8000/predict

=== Avec authentification Bearer ===
curl -s -H "Authorization: Bearer mon_token_jwt" \\
     http://localhost:8000/predict

=== Vérifier juste le status HTTP ===
curl -s -o /dev/null -w "%{http_code}" http://localhost:8000/health

=== GET /metrics Prometheus ===
curl -s http://localhost:8000/metrics | grep http_requests

=== Envoyer un fichier ===
curl -s -X POST -F "file=@data/commandes.csv" http://localhost:8000/upload

=== Sauvegarder la réponse ===
curl -s http://localhost:8000/predict -o response.json
cat response.json | python3 -m json.tool

=== Verbose (débogage) ===
curl -v http://localhost:8000/health 2>&1 | head -40

=== Mesurer les performances ===
curl -s -o /dev/null -w "DNS: %{time_namelookup}s | Connexion: %{time_connect}s | Total: %{time_total}s\n" \\
     http://localhost:8000/health

=== Boucle : tester plusieurs fois ===
for i in $(seq 1 10); do
    curl -s http://localhost:8000/health -w " → status: %{http_code}\n"
done
""")

---
<a id="12-kubernetes"></a>
## 12. Kubernetes — ressources essentielles

### Hiérarchie des ressources
```
Cluster
└── Namespace (isolation logique)
    ├── ConfigMap   (configuration non sensible)
    ├── Secret      (données sensibles)
    ├── PersistentVolume (stockage physique)
    ├── PersistentVolumeClaim (demande de stockage)
    ├── Service     (exposition réseau stable)
    └── Deployment  (gestion des Pods)
        └── Pod     (container(s) + volumes)
```


In [ ]:
# 12.1 — Namespace

namespace_yaml = """
apiVersion: v1
kind: Namespace
metadata:
  name: techshop-dev
  labels:
    env: development
    team: backend
"""

print("Namespace YAML :")
print(namespace_yaml)

print("""
=== Commandes Namespace ===
kubectl create namespace techshop-dev
kubectl get namespaces
kubectl get all -n techshop-dev

# Changer le namespace par défaut
kubectl config set-context --current --namespace=techshop-dev

# Supprimer (supprime TOUT ce qu'il contient !)
kubectl delete namespace techshop-dev
""")

In [ ]:
# 12.2 — PersistentVolume + PersistentVolumeClaim

pv_pvc_yaml = """
# ── PersistentVolume (admin) ──────────────────────────────────────
apiVersion: v1
kind: PersistentVolume
metadata:
  name: techshop-pv
spec:
  capacity:
    storage: 5Gi
  accessModes:
    - ReadWriteOnce
  persistentVolumeReclaimPolicy: Retain
  storageClassName: standard
  hostPath:
    path: /mnt/data/techshop    # nœud unique (dev)

---
# ── PersistentVolumeClaim (développeur) ──────────────────────────
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: techshop-pvc
  namespace: techshop-dev
spec:
  accessModes:
    - ReadWriteOnce
  storageClassName: standard
  resources:
    requests:
      storage: 2Gi
"""
print(pv_pvc_yaml)

print("""
=== Commandes PV/PVC ===
kubectl get pv
kubectl get pvc -n techshop-dev
kubectl describe pvc techshop-pvc -n techshop-dev

# États possibles du PVC :
# Pending   → aucun PV disponible correspondant
# Bound     → PVC associé à un PV (prêt)
# Lost      → le PV a été supprimé
""")

In [ ]:
# 12.3 — ConfigMap

configmap_yaml = """
apiVersion: v1
kind: ConfigMap
metadata:
  name: techshop-config
  namespace: techshop-dev
data:
  # Variables d'environnement
  APP_ENV: "development"
  DB_HOST: "mysql-service"
  DB_PORT: "3306"
  DB_NAME: "techshop"
  LOG_LEVEL: "INFO"

  # Fichier de configuration (clé = nom du fichier)
  prometheus.yml: |
    global:
      scrape_interval: 15s
    scrape_configs:
      - job_name: 'techshop-api'
        static_configs:
          - targets: ['api-service:80']    # port du Service (→ 8000 du pod)
      - job_name: 'mysql'
        static_configs:
          - targets: ['mysql-exporter:9104']
"""
print("ConfigMap YAML :")
print(configmap_yaml)

print("""
=== Utilisation dans un Deployment ===
# Toutes les clés comme variables d'env :
envFrom:
  - configMapRef:
      name: techshop-config

# Une clé spécifique :
env:
  - name: DB_HOST
    valueFrom:
      configMapKeyRef:
        name: techshop-config
        key: DB_HOST

# Monter comme fichier :
volumes:
  - name: cfg
    configMap:
      name: techshop-config
      items:
        - key: prometheus.yml
          path: prometheus.yml

=== Commandes ConfigMap ===
kubectl apply -f configmap.yaml
kubectl get configmap -n techshop-dev
kubectl describe configmap techshop-config -n techshop-dev
kubectl create configmap app-cfg --from-file=prometheus.yml
kubectl create configmap env-cfg --from-literal=APP_ENV=dev --from-literal=DB_HOST=localhost
""")

In [ ]:
# 12.4 — Service

service_yaml = """
# ── ClusterIP — interne uniquement (défaut) ───────────────────────
apiVersion: v1
kind: Service
metadata:
  name: api-service
  namespace: techshop-dev
spec:
  selector:
    app: techshop-api         # cible les pods avec ce label
  ports:
    - port: 80                # port du service (accessible via api-service:80)
      targetPort: 8000        # port du container
  type: ClusterIP

---
# ── NodePort — accès depuis l'extérieur ───────────────────────────
apiVersion: v1
kind: Service
metadata:
  name: api-nodeport
  namespace: techshop-dev
spec:
  selector:
    app: techshop-api
  ports:
    - port: 80
      targetPort: 8000
      nodePort: 30080         # IP_DU_NOEUD:30080 (plage 30000-32767)
  type: NodePort
"""
print(service_yaml)

print("""
=== Commandes Service ===
kubectl get services -n techshop-dev
kubectl describe service api-service -n techshop-dev

# Accéder depuis minikube
minikube service api-nodeport -n techshop-dev --url

# Port-forward (debug local)
kubectl port-forward service/api-service 8080:80 -n techshop-dev
curl http://localhost:8080/health
""")

In [ ]:
# 12.5 — Deployment complet

deployment_yaml = """
apiVersion: apps/v1
kind: Deployment
metadata:
  name: techshop-api
  namespace: techshop-dev
  labels:
    app: techshop-api
    version: v1.2.0
spec:
  replicas: 2
  selector:
    matchLabels:
      app: techshop-api

  strategy:
    type: RollingUpdate
    rollingUpdate:
      maxSurge: 1
      maxUnavailable: 0   # zero-downtime

  template:
    metadata:
      labels:
        app: techshop-api
    spec:
      containers:
        - name: api
          image: moncompte/techshop-api:v1.2.0
          imagePullPolicy: IfNotPresent
          ports:
            - containerPort: 8000

          envFrom:
            - configMapRef:
                name: techshop-config
            - secretRef:
                name: techshop-secrets

          volumeMounts:
            - name: models
              mountPath: /app/models

          resources:
            requests: {memory: "128Mi", cpu: "100m"}
            limits:   {memory: "512Mi", cpu: "500m"}

          livenessProbe:
            httpGet: {path: /health, port: 8000}
            initialDelaySeconds: 15
            periodSeconds: 20

          readinessProbe:
            httpGet: {path: /ready, port: 8000}
            initialDelaySeconds: 5
            periodSeconds: 10

      volumes:
        - name: models
          persistentVolumeClaim:
            claimName: techshop-pvc
"""
print(deployment_yaml)

print("""
=== Commandes Deployment ===
kubectl apply -f deployment.yaml
kubectl get deployments -n techshop-dev
kubectl describe deployment techshop-api -n techshop-dev
kubectl get pods -n techshop-dev -l app=techshop-api

# Rolling update (nouvelle image)
kubectl set image deployment/techshop-api api=moncompte/techshop-api:v1.3.0 -n techshop-dev
kubectl rollout status deployment/techshop-api -n techshop-dev

# Historique + rollback
kubectl rollout history deployment/techshop-api -n techshop-dev
kubectl rollout undo  deployment/techshop-api -n techshop-dev

# Scaler
kubectl scale deployment techshop-api --replicas=4 -n techshop-dev
""")

In [ ]:
# 12.6 — mysqld_exporter dans Kubernetes (Deployment + Service)

mysql_exporter_k8s_yaml = """
apiVersion: apps/v1
kind: Deployment
metadata:
  name: mysql-exporter
  namespace: techshop-dev
  labels:
    app: mysql-exporter
spec:
  replicas: 1                     # 1 seul : sinon métriques en double
  selector:
    matchLabels:
      app: mysql-exporter
  template:
    metadata:
      labels:
        app: mysql-exporter
    spec:
      containers:
        - name: mysql-exporter
          image: prom/mysqld-exporter:v0.15.1
          args:
            - --mysqld.address=mysql-service:3306   # Service MySQL (cf. DB_HOST du ConfigMap)
            - --mysqld.username=exporter
          env:
            - name: MYSQLD_EXPORTER_PASSWORD
              valueFrom:
                secretKeyRef:
                  name: techshop-secrets
                  key: EXPORTER_PASSWORD
          ports:
            - containerPort: 9104
          resources:
            requests: {memory: "32Mi", cpu: "10m"}
            limits:   {memory: "64Mi", cpu: "100m"}
          livenessProbe:
            httpGet: {path: /, port: 9104}
            initialDelaySeconds: 10
            periodSeconds: 30

---
apiVersion: v1
kind: Service
metadata:
  name: mysql-exporter            # = cible 'mysql-exporter:9104' du prometheus.yml
  namespace: techshop-dev
spec:
  selector:
    app: mysql-exporter
  ports:
    - port: 9104
      targetPort: 9104
  type: ClusterIP
"""
print(mysql_exporter_k8s_yaml)

print("""
=== Prérequis ===
# 1. Le mot de passe dans le Secret (clé EXPORTER_PASSWORD)
kubectl create secret generic techshop-secrets -n techshop-dev \\
  --from-literal=EXPORTER_PASSWORD=exporter_pwd_a_changer \\
  --dry-run=client -o yaml | kubectl apply -f -

# 2. Le compte MySQL (même SQL que 8.3 bis)
kubectl exec -it deploy/mysql -n techshop-dev -- mysql -uroot -p
  CREATE USER 'exporter'@'%' IDENTIFIED BY 'exporter_pwd_a_changer' WITH MAX_USER_CONNECTIONS 3;
  GRANT PROCESS, REPLICATION CLIENT, SELECT ON *.* TO 'exporter'@'%';

=== Commandes ===
kubectl apply -f mysql-exporter.yaml
kubectl get pods -n techshop-dev -l app=mysql-exporter
kubectl logs -l app=mysql-exporter -n techshop-dev
kubectl port-forward service/mysql-exporter 9104:9104 -n techshop-dev
curl -s localhost:9104/metrics | grep mysql_up    # mysql_up 1 = OK
""")

---
<a id="13-prometheus"></a>
## 13. Prometheus — configuration & PromQL

### Architecture
```
FastAPI (expose /metrics)      mysqld_exporter :9104       node_exporter :9100
                               (lit MySQL, traduit         (CPU, RAM, disque
                                en métriques)               de la machine)
        ↑                              ↑                            ↑
        └──────────── scrape toutes les 15s (HTTP GET /metrics) ─────┘
Prometheus (stocke les séries temporelles)
    ↑ requêtes PromQL
Grafana (dashboards)
```

> MySQL ne parle pas le format Prometheus : un **exporter** est un service à part qui se connecte
> à la base (compte en lecture seule) et expose ses statistiques sur `/metrics`.

### Types de métriques
| Type | Description | Exemple |
|------|-------------|---------|
| **Counter** | Toujours croissant | `http_requests_total` |
| **Gauge** | Peut monter/descendre | `http_requests_in_progress` |
| **Histogram** | Distribution + buckets | `http_request_duration_seconds` |
| **Summary** | count + sum (pas de quantiles en Python) | `techshop_etl_duree_seconds` |
| **Info** | Métadonnées constantes (labels) | `techshop_modele_info` |
| **Enum** | Un état actif parmi N | `techshop_modele_etat` |


In [ ]:
# 13.1 — prometheus_fastapi_instrumentator

instrumentator_code = """
# Installation :
# pip install prometheus-fastapi-instrumentator

from fastapi import FastAPI
from prometheus_fastapi_instrumentator import Instrumentator

app = FastAPI()

# ── Configuration basique (ajoute GET /metrics) ───────────────────
Instrumentator().instrument(app=app).expose(app=app)

# ── Configuration avancée ────────────────────────────────────────
instrumentator = Instrumentator(
    should_group_status_codes=True,          # regroupe 2xx, 4xx, 5xx
    should_ignore_untemplated=True,          # ignore /docs, /openapi.json
    should_respect_env_var=True,             # respecter ENABLE_METRICS=true
    should_instrument_requests_inprogress=True,  # gauge des requêtes en cours
    inprogress_name="http_requests_in_progress",
    inprogress_labels=True,
)

# Métriques supplémentaires personnalisées
from prometheus_client import Counter, Gauge, Histogram

PREDICTIONS_TOTAL = Counter(
    name="techshop_predictions_total",
    documentation="Nombre total de prédictions",
    labelnames=["segment", "risque"]
)
CHURN_PROBA_HIST = Histogram(
    name="techshop_churn_proba",
    documentation="Distribution des probabilités de churn",
    buckets=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
)
MODEL_LOADED = Gauge(name="techshop_model_loaded", documentation="1 si le modèle est chargé")

@app.post(path="/predict")
def predict(features: ClientFeatures):
    proba = model.predict_proba(X=X)[0, 1]

    # Incrémenter les métriques
    PREDICTIONS_TOTAL.labels(
        segment=features.segment,
        risque="Élevé" if proba > 0.6 else "Moyen" if proba > 0.3 else "Faible"
    ).inc()
    CHURN_PROBA_HIST.observe(amount=proba)

    return {"proba": proba}
"""
print(instrumentator_code)

In [ ]:
# 13.1 bis — Métriques custom avec prometheus_client (métier, ML, jobs batch)

custom_metrics_code = """
# metrics.py — métriques métier / ML / jobs (prometheus_client)
# pip install prometheus-client
#
# Déjà fournies par Instrumentator (NE PAS les recréer) :
#   http_requests_total, http_request_duration_seconds, http_request_size_bytes,
#   http_response_size_bytes, http_requests_inprogress
# Déjà fournies par prometheus_client lui-même : process_* (CPU, RAM), python_gc_*
#
# Conventions de nommage :
#   - préfixe applicatif   : techshop_...
#   - unité en suffixe     : _seconds, _bytes, _euros
#   - Counter → _total     : prometheus_client l'ajoute à l'exposition
#   - labels à FAIBLE cardinalité (segment, statut…) — jamais client_id, email, uuid !

import time
from prometheus_client import Counter, Gauge, Histogram, Summary, Info, Enum

# ════════════════════════════════════════════════════════════════════
# 1. COUNTER — ne fait que monter (repart à 0 au redémarrage)
# ════════════════════════════════════════════════════════════════════

# Événement métier avec labels
COMMANDES_TOTAL = Counter(
    name="techshop_commandes_total",
    documentation="Commandes créées",
    labelnames=["canal", "statut"],           # canal = web/mobile, statut = ok/refusee
)
COMMANDES_TOTAL.labels(canal="web", statut="ok").inc()

# Counter incrémenté d'un MONTANT (pas seulement +1)
CA_TOTAL = Counter(
    name="techshop_chiffre_affaires_euros_total",
    documentation="Chiffre d'affaires cumulé en euros",
    labelnames=["canal"],
)
CA_TOTAL.labels(canal="web").inc(amount=149.90)

# Compter les exceptions automatiquement (décorateur)
ERREURS_PREDICTION = Counter(
    name="techshop_prediction_erreurs_total",
    documentation="Prédictions ayant levé une exception",
)

@ERREURS_PREDICTION.count_exceptions(exception=ValueError)
def predire(features: dict) -> float:
    if "age" not in features:
        raise ValueError("feature 'age' manquante")
    return 0.42

# Appels à une API externe (dépendance)
APPELS_EXTERNES = Counter(
    name="techshop_api_externe_appels_total",
    documentation="Appels vers les API partenaires",
    labelnames=["api", "resultat"],           # resultat = succes/timeout/erreur
)
APPELS_EXTERNES.labels(api="taux_change", resultat="timeout").inc()

# ════════════════════════════════════════════════════════════════════
# 2. GAUGE — valeur instantanée qui monte ET descend
# ════════════════════════════════════════════════════════════════════

# Taille d'une file d'attente / d'un stock
STOCK = Gauge(
    name="techshop_stock_produits",
    documentation="Unités en stock par produit",
    labelnames=["categorie"],
)
STOCK.labels(categorie="ordinateurs").set(value=37)
STOCK.labels(categorie="ordinateurs").dec(amount=1)   # une vente

# Valeur calculée à chaque scrape (pas besoin de la mettre à jour)
PANIERS_ACTIFS = {"a1": [], "b2": []}
NB_PANIERS = Gauge(
    name="techshop_paniers_actifs",
    documentation="Paniers ouverts en mémoire",
)
NB_PANIERS.set_function(f=lambda: len(PANIERS_ACTIFS))

# Horodatage du dernier rechargement du modèle (→ alerte si trop vieux)
MODELE_RECHARGE_TS = Gauge(
    name="techshop_modele_dernier_chargement_timestamp_seconds",
    documentation="Timestamp Unix du dernier chargement du modèle",
)
MODELE_RECHARGE_TS.set_to_current_time()

# Jobs batch en cours (≠ http_requests_inprogress : ici ce sont des traitements internes)
JOBS_EN_COURS = Gauge(
    name="techshop_jobs_en_cours",
    documentation="Traitements batch en cours d'exécution",
    labelnames=["job"],
)

@JOBS_EN_COURS.labels(job="recalcul_segments").track_inprogress()
def recalculer_segments():
    time.sleep(0.01)

# Data drift : moyenne d'une feature en production (à comparer à l'entraînement)
FEATURE_MOYENNE = Gauge(
    name="techshop_feature_moyenne",
    documentation="Moyenne glissante des features reçues en production",
    labelnames=["feature"],
)
FEATURE_MOYENNE.labels(feature="age").set(value=41.7)
FEATURE_MOYENNE.labels(feature="ca_total").set(value=812.4)

# ════════════════════════════════════════════════════════════════════
# 3. HISTOGRAM — distribution (buckets) → percentiles côté PromQL
# ════════════════════════════════════════════════════════════════════

# Temps d'INFÉRENCE du modèle seul (≠ latence HTTP complète d'Instrumentator)
INFERENCE_SECONDES = Histogram(
    name="techshop_modele_inference_seconds",
    documentation="Durée de model.predict_proba()",
    labelnames=["modele"],
    buckets=[0.001, 0.005, 0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1.0],
)

def predire_chronometre(X):
    with INFERENCE_SECONDES.labels(modele="churn_v1").time():   # context manager
        time.sleep(0.002)                                        # model.predict_proba(X=X)
        return 0.42

# Distribution d'une valeur MÉTIER : montant des paniers
MONTANT_PANIER = Histogram(
    name="techshop_panier_montant_euros",
    documentation="Montant des paniers validés",
    buckets=[10, 25, 50, 100, 250, 500, 1000, 2500],
)
MONTANT_PANIER.observe(amount=149.90)

# ════════════════════════════════════════════════════════════════════
# 4. SUMMARY — count + sum (prometheus_client Python ne calcule PAS de quantiles)
#    → utile pour une MOYENNE : rate(_sum) / rate(_count)
# ════════════════════════════════════════════════════════════════════
ETL_LIGNES = Summary(
    name="techshop_etl_lignes_traitees",
    documentation="Nombre de lignes traitées par exécution de l'ETL",
)
ETL_LIGNES.observe(amount=12500)

ETL_DUREE = Summary(
    name="techshop_etl_duree_seconds",
    documentation="Durée d'une exécution de l'ETL",
)

@ETL_DUREE.time()                             # décorateur
def lancer_etl():
    time.sleep(0.01)

# ════════════════════════════════════════════════════════════════════
# 5. INFO & ENUM — métadonnées et état
# ════════════════════════════════════════════════════════════════════

# Info : paires clé/valeur constantes (exposé comme un gauge = 1)
MODELE_INFO = Info(
    name="techshop_modele",
    documentation="Version et caractéristiques du modèle servi",
)
MODELE_INFO.info(val={"version": "1.2.0", "algo": "RandomForest", "entraine_le": "2026-10-01"})

# Enum : un seul état actif à la fois (1 pour l'état courant, 0 pour les autres)
MODELE_ETAT = Enum(
    name="techshop_modele_etat",
    documentation="État du modèle",
    states=["chargement", "pret", "erreur"],
)
MODELE_ETAT.state(state="pret")

# ════════════════════════════════════════════════════════════════════
# 6. EXPOSITION
# ════════════════════════════════════════════════════════════════════
# a) Dans l'API FastAPI : RIEN à faire. Instrumentator().expose(app=app) sert le
#    REGISTRY par défaut → ces métriques apparaissent sur le même GET /metrics.
#
# b) Dans un worker / script long SANS FastAPI : petit serveur HTTP dédié
#    from prometheus_client import start_http_server
#    start_http_server(port=8001)            # → http://worker:8001/metrics
#
# c) Job batch court (cron) : il finit avant d'être scrapé → on POUSSE vers le Pushgateway
def pousser_metriques_batch(nb_lignes: int):
    from prometheus_client import CollectorRegistry, push_to_gateway
    registry = CollectorRegistry()            # registre dédié (pas le global)
    derniere_reussite = Gauge(
        name="techshop_etl_derniere_reussite_timestamp_seconds",
        documentation="Fin de la dernière exécution réussie de l'ETL",
        registry=registry,
    )
    lignes = Gauge(
        name="techshop_etl_lignes_derniere_execution",
        documentation="Lignes traitées lors de la dernière exécution",
        registry=registry,
    )
    lignes.set(value=nb_lignes)
    derniere_reussite.set_to_current_time()
    push_to_gateway(gateway="pushgateway:9091", job="etl_nuit", registry=registry)
"""
print(custom_metrics_code)

print("""
=== Ce que Prometheus lit sur GET /metrics (extrait) ===
techshop_commandes_total{canal="web",statut="ok"} 1.0
techshop_chiffre_affaires_euros_total{canal="web"} 149.9
techshop_stock_produits{categorie="ordinateurs"} 36.0
techshop_modele_inference_seconds_bucket{le="0.005",modele="churn_v1"} 1.0
techshop_modele_inference_seconds_count{modele="churn_v1"} 1.0
techshop_modele_inference_seconds_sum{modele="churn_v1"} 0.002
techshop_modele_info{algo="RandomForest",entraine_le="2026-10-01",version="1.2.0"} 1.0
techshop_modele_etat{techshop_modele_etat="pret"} 1.0
→ un Histogram/Summary produit plusieurs séries : _bucket (Histogram seulement), _count, _sum

=== PromQL — exploiter ces métriques ===

# Commandes validées par minute, par canal
sum by (canal) (rate(techshop_commandes_total{statut="ok"}[5m])) * 60

# Taux de commandes refusées (ratio de deux Counters)
sum(rate(techshop_commandes_total{statut="refusee"}[15m]))
  / sum(rate(techshop_commandes_total[15m]))

# Chiffre d'affaires de la dernière heure, par canal
sum by (canal) (increase(techshop_chiffre_affaires_euros_total[1h]))

# Part des appels à l'API de taux de change en timeout
sum(rate(techshop_api_externe_appels_total{api="taux_change",resultat="timeout"}[5m]))
  / sum(rate(techshop_api_externe_appels_total{api="taux_change"}[5m]))

# Produits presque en rupture
techshop_stock_produits < 5

# Âge du modèle chargé, en heures
(time() - techshop_modele_dernier_chargement_timestamp_seconds) / 3600

# p95 du temps d'inférence (Histogram → histogram_quantile sur les _bucket)
histogram_quantile(0.95, sum by (le) (rate(techshop_modele_inference_seconds_bucket[5m])))

# Panier médian / panier moyen sur 1h
histogram_quantile(0.5, sum by (le) (rate(techshop_panier_montant_euros_bucket[1h])))
rate(techshop_panier_montant_euros_sum[1h]) / rate(techshop_panier_montant_euros_count[1h])

# Durée moyenne de l'ETL (Summary → _sum / _count)
rate(techshop_etl_duree_seconds_sum[1d]) / rate(techshop_etl_duree_seconds_count[1d])

# Data drift : écart à la moyenne vue à l'entraînement (ici 40.2 ans)
abs(techshop_feature_moyenne{feature="age"} - 40.2)

# Version du modèle actuellement servie (Info → lire les labels)
techshop_modele_info

=== Règles d'alerte (extrait — fichier complet en 13.5) ===
groups:
  - name: techshop-metier
    rules:
      - alert: ModeleNonPret
        expr: techshop_modele_etat{techshop_modele_etat="pret"} == 0
        for: 2m
        labels: {severity: critical}
      - alert: InferenceLente
        expr: histogram_quantile(0.95, sum by (le) (rate(techshop_modele_inference_seconds_bucket[5m]))) > 0.25
        for: 10m
        labels: {severity: warning}
      - alert: DriftAge
        expr: abs(techshop_feature_moyenne{feature="age"} - 40.2) > 5
        for: 1h
        labels: {severity: warning}
      - alert: ETLEnRetard              # pas de succès depuis plus de 26h
        expr: time() - techshop_etl_derniere_reussite_timestamp_seconds > 26 * 3600
        labels: {severity: critical}

=== Pushgateway : à ajouter dans prometheus.yml (+ service prom/pushgateway:9091) ===
  - job_name: 'pushgateway'
    honor_labels: true                  # garde le label job="etl_nuit" poussé par le script
    static_configs:
      - targets: ['pushgateway:9091']

=== Choisir le bon type ===
Ça ne fait que monter (événements, montants cumulés) ......... Counter   → rate() / increase()
Valeur du moment (stock, file, timestamp, état) .............. Gauge     → valeur brute
Distribution, percentiles (latence, montants) ................ Histogram → histogram_quantile()
Moyenne seulement (count + sum) .............................. Summary   → _sum / _count
Métadonnées constantes (version, algo) ....................... Info
Un état parmi N .............................................. Enum
""")

In [ ]:
# 13.2 — prometheus.yml (configuration du serveur Prometheus)

prom_config = """
# config/prometheus.yml

global:
  scrape_interval:     15s    # fréquence de scraping
  evaluation_interval: 15s    # fréquence d'évaluation des règles
  external_labels:
    monitor: techshop-monitor

# ── Règles (chemins relatifs à ce fichier → /etc/prometheus/rules/) ──
rule_files:
  - "rules/recording_rules.yml"     # d'abord les séries pré-calculées…
  - "rules/alert_rules.yml"         # …utilisées par les alertes

# ── Alertmanager ──────────────────────────────────────────────────
alerting:
  alertmanagers:
    - static_configs:
        - targets: ['alertmanager:9093']

# ── Configuration des cibles à scraper ───────────────────────────
scrape_configs:

  # Prometheus lui-même
  - job_name: 'prometheus'
    static_configs:
      - targets: ['localhost:9090']

  # API FastAPI TechShop
  - job_name: 'techshop-api'
    metrics_path: '/metrics'
    scrape_interval: 10s
    static_configs:
      - targets: ['api:8000']     # nom du service docker-compose (en K8s : 'api-service:80')
        labels:
          env: development
          service: techshop-api

  # MySQL (via mysqld_exporter — service 'mysql-exporter' du compose, cf. 8.3)
  - job_name: 'mysql'
    static_configs:
      - targets: ['mysql-exporter:9104']

  # Node (métriques machine)
  - job_name: 'node'
    static_configs:
      - targets: ['node-exporter:9100']

  # Alertmanager (sa propre santé : notifications envoyées / en échec)
  - job_name: 'alertmanager'
    static_configs:
      - targets: ['alertmanager:9093']

  # Pushgateway (métriques poussées par les jobs batch, ex. ETL de nuit)
  - job_name: 'pushgateway'
    honor_labels: true                # garde le label job="etl_nuit" poussé par le script
    static_configs:
      - targets: ['pushgateway:9091']
"""
print(prom_config)

In [ ]:
# 13.2 bis — Métriques MySQL utiles (mysqld_exporter)

print("""
=== PromQL — MySQL ===

# L'exporter arrive-t-il à joindre MySQL ? (1 = oui, 0 = non)
mysql_up

# Prometheus arrive-t-il à joindre l'exporter ? (job 'mysql')
up{job="mysql"}

# Requêtes par seconde
rate(mysql_global_status_queries[5m])

# Connexions ouvertes / max autorisé
mysql_global_status_threads_connected
mysql_global_status_threads_connected / mysql_global_variables_max_connections

# Requêtes lentes par seconde
rate(mysql_global_status_slow_queries[5m])

# Uptime du serveur MySQL (secondes)
mysql_global_status_uptime

=== Exemple de règle d'alerte (extrait — fichier complet en 13.5) ===
groups:
  - name: mysql
    rules:
      - alert: MySQLDown
        expr: mysql_up == 0
        for: 1m
        labels: {severity: critical}
        annotations:
          summary: "MySQL injoignable depuis mysqld_exporter"

=== Grafana ===
Dashboard communautaire "MySQL Overview" : Dashboards → New → Import → ID 7362
""")

In [ ]:
# 13.3 — PromQL : langage de requête Prometheus

print("""
=== PromQL — Requêtes essentielles ===

# ── Compteurs (Counter) ─────────────────────────────────────────────
# Nombre de requêtes HTTP total
http_requests_total

# Taux de requêtes par seconde (sur 5 minutes)
rate(http_requests_total[5m])

# Taux par endpoint (sum by)
sum by(handler) (rate(http_requests_total[5m]))

# Requêtes en erreur (label 'status' d'Instrumentator, regroupé en 2xx/4xx/5xx)
sum(rate(http_requests_total{status="5xx"}[5m]))

# ── Latence (Histogram) ─────────────────────────────────────────────
# http_request_duration_seconds : buckets larges (0.1, 0.5, 1) mais label handler
# http_request_duration_highr_seconds : buckets fins, sans label → percentiles précis
# Latence médiane (p50)
histogram_quantile(0.50, sum(rate(http_request_duration_seconds_bucket[5m])) by (le))

# Latence p95
histogram_quantile(0.95, sum(rate(http_request_duration_seconds_bucket[5m])) by (le))

# Latence p99 par endpoint
histogram_quantile(0.99,
  sum(rate(http_request_duration_seconds_bucket[5m])) by (le, handler)
)

# ── Métriques custom TechShop ────────────────────────────────────────
# Total des prédictions churn par segment
techshop_predictions_total

# Taux de prédictions par risque
sum by(risque) (rate(techshop_predictions_total[1m]))

# Distribution des probabilités de churn (p50, p95)
histogram_quantile(0.50, rate(techshop_churn_proba_bucket[5m]))
histogram_quantile(0.95, rate(techshop_churn_proba_bucket[5m]))

# ── Combinaisons ────────────────────────────────────────────────────
# Taux d'erreur (%)
100 * sum(rate(http_requests_total{status="5xx"}[5m]))
    / sum(rate(http_requests_total[5m]))

# Disponibilité (uptime rate sur 30 jours)
avg_over_time(up{job="techshop-api"}[30d]) * 100

# ── Séries pré-calculées (recording rules, 13.4) : plus simple et plus rapide ──
job:http_requests_errors:ratio_rate5m{job="techshop-api"}
job:http_request_duration_seconds:p95_5m
instance:node_cpu_utilisation:rate5m
""")

### Alerting : la chaîne complète

```
             recording_rules.yml            alert_rules.yml               alertmanager.yml
métriques ──► séries pré-calculées ──► condition vraie pendant `for` ──► regroupe, route, inhibe ──► email / Slack / webhook
(scrape)      (job:...:rate5m)          pending → firing                   (Alertmanager :9093)
```

| Fichier | Lu par | Rôle |
|---|---|---|
| `rules/recording_rules.yml` | Prometheus | calcule des requêtes coûteuses à l'avance → nouvelles séries |
| `rules/alert_rules.yml` | Prometheus | définit **quand** une alerte se déclenche |
| `alertmanager/alertmanager.yml` | Alertmanager | définit **qui** est prévenu, **comment** et **quand se taire** |
| `rules/tests/rules_test.yml` | promtool | tests unitaires des règles (séries fictives) |

> Prometheus **n'envoie aucun email** : il ne fait qu'évaluer les règles et pousser les alertes « firing »
> vers Alertmanager. Tous les fichiers ci-dessous ont été validés avec `promtool` et `amtool` (cf. 13.7).

In [ ]:
# 13.4 — Règles d'enregistrement (config/rules/recording_rules.yml)

recording_rules_yaml = """
# config/rules/recording_rules.yml
# Règles d'ENREGISTREMENT : Prometheus calcule ces requêtes à chaque évaluation
# et stocke le résultat comme une NOUVELLE série → dashboards et alertes plus rapides,
# et une seule définition (pas de PromQL copié-collé partout).
#
# Convention de nommage : niveau:métrique:opérations
#   niveau     = labels conservés après agrégation (job, job_handler, instance…)
#   métrique   = nom de la métrique d'origine (sans _total)
#   opérations = ce qui a été appliqué (rate5m, ratio_rate5m, p95_5m…)

groups:

  # ── API FastAPI (métriques de prometheus_fastapi_instrumentator) ─────────
  - name: techshop-api.recording
    interval: 30s                       # sinon : evaluation_interval global (15s)
    rules:
      # Débit total et par endpoint (req/s)
      - record: job:http_requests:rate5m
        expr: sum by (job) (rate(http_requests_total[5m]))

      - record: job_handler:http_requests:rate5m
        expr: sum by (job, handler) (rate(http_requests_total[5m]))

      # Part des réponses 5xx (status regroupé "5xx" car should_group_status_codes=True)
      - record: job:http_requests_errors:ratio_rate5m
        expr: |
          sum by (job) (rate(http_requests_total{status="5xx"}[5m]))
            /
          sum by (job) (rate(http_requests_total[5m]))

      # Latences : http_request_duration_highr_seconds a des buckets fins (sans label handler)
      - record: job:http_request_duration_seconds:p50_5m
        expr: histogram_quantile(0.50, sum by (job, le) (rate(http_request_duration_highr_seconds_bucket[5m])))

      - record: job:http_request_duration_seconds:p95_5m
        expr: histogram_quantile(0.95, sum by (job, le) (rate(http_request_duration_highr_seconds_bucket[5m])))

      - record: job:http_request_duration_seconds:p99_5m
        expr: histogram_quantile(0.99, sum by (job, le) (rate(http_request_duration_highr_seconds_bucket[5m])))

      # Latence moyenne par endpoint (http_request_duration_seconds a le label handler)
      - record: job_handler:http_request_duration_seconds:mean_5m
        expr: |
          sum by (job, handler) (rate(http_request_duration_seconds_sum[5m]))
            /
          sum by (job, handler) (rate(http_request_duration_seconds_count[5m]))

  # ── Métier & ML (métriques custom prometheus_client, cf. 13.1 bis) ───────
  - name: techshop-metier.recording
    interval: 1m
    rules:
      - record: canal:techshop_commandes:rate5m
        expr: sum by (canal) (rate(techshop_commandes_total{statut="ok"}[5m]))

      - record: techshop_commandes_refusees:ratio_rate15m
        expr: |
          sum(rate(techshop_commandes_total{statut="refusee"}[15m]))
            /
          sum(rate(techshop_commandes_total[15m]))

      - record: canal:techshop_chiffre_affaires_euros:increase1h
        expr: sum by (canal) (increase(techshop_chiffre_affaires_euros_total[1h]))

      - record: techshop_panier_montant_euros:mean1h
        expr: |
          sum(rate(techshop_panier_montant_euros_sum[1h]))
            /
          sum(rate(techshop_panier_montant_euros_count[1h]))

      - record: modele:techshop_modele_inference_seconds:p95_5m
        expr: histogram_quantile(0.95, sum by (modele, le) (rate(techshop_modele_inference_seconds_bucket[5m])))

      - record: techshop_prediction_erreurs:rate5m
        expr: sum(rate(techshop_prediction_erreurs_total[5m]))

      - record: api:techshop_api_externe_timeouts:ratio_rate5m
        expr: |
          sum by (api) (rate(techshop_api_externe_appels_total{resultat="timeout"}[5m]))
            /
          sum by (api) (rate(techshop_api_externe_appels_total[5m]))

      # Écart (en %) entre la moyenne d'une feature en prod et celle vue à l'entraînement
      # Les références d'entraînement sont des constantes → une série par feature
      # (labels: ajoute des labels fixes à la série enregistrée)
      - record: feature:techshop_feature_reference
        expr: vector(40.2)
        labels:
          feature: age
      - record: feature:techshop_feature_reference
        expr: vector(790)
        labels:
          feature: ca_total

      - record: feature:techshop_feature_drift:ratio
        expr: |
          abs(
            max by (feature) (techshop_feature_moyenne)
              - on (feature)
            feature:techshop_feature_reference
          )
            / on (feature)
          feature:techshop_feature_reference

  # ── Infrastructure (node_exporter + mysqld_exporter) ─────────────────────
  - name: techshop-infra.recording
    interval: 30s
    rules:
      # Part du CPU utilisée (0 → 1), toutes CPU confondues
      - record: instance:node_cpu_utilisation:rate5m
        expr: 1 - avg by (instance) (rate(node_cpu_seconds_total{mode="idle"}[5m]))

      # Part de la mémoire encore disponible (0 → 1)
      - record: instance:node_memory_disponible:ratio
        expr: node_memory_MemAvailable_bytes / node_memory_MemTotal_bytes

      # Part d'espace libre par système de fichiers réel (pas tmpfs, overlay…)
      - record: instance_mountpoint:node_filesystem_libre:ratio
        expr: |
          node_filesystem_avail_bytes{fstype!~"tmpfs|overlay|squashfs"}
            /
          node_filesystem_size_bytes{fstype!~"tmpfs|overlay|squashfs"}

      - record: instance:mysql_global_status_queries:rate5m
        expr: rate(mysql_global_status_queries[5m])

      - record: instance:mysql_connexions:ratio
        expr: mysql_global_status_threads_connected / mysql_global_variables_max_connections
"""
print(recording_rules_yaml)

In [ ]:
# 13.5 — Règles d'alerte (config/rules/alert_rules.yml)

alert_rules_yaml = """
# config/rules/alert_rules.yml
# Règles d'ALERTE : évaluées en continu par Prometheus.
#   expr        → condition (s'appuie sur les séries de recording_rules.yml)
#   for         → durée pendant laquelle la condition doit rester vraie (évite les faux positifs)
#                 pending (condition vraie, durée non atteinte) → firing (envoyée à Alertmanager)
#   labels      → servent au ROUTAGE dans Alertmanager (severity, team)
#   annotations → texte lisible ; {{ $labels.x }} et {{ $value }} sont remplacés

groups:

  # ── Disponibilité des cibles ─────────────────────────────────────────────
  - name: techshop-disponibilite
    rules:
      - alert: CibleInjoignable
        expr: up == 0
        for: 1m
        labels:
          severity: critical
          team: ops
        annotations:
          summary: "{{ $labels.job }} injoignable ({{ $labels.instance }})"
          description: "Prometheus n'arrive plus à scraper {{ $labels.instance }} depuis plus d'une minute."

      - alert: APIAbsente
        expr: absent(up{job="techshop-api"})      # le job a disparu de la config / de la découverte
        for: 5m
        labels:
          severity: critical
          team: backend
        annotations:
          summary: "Aucune cible techshop-api n'est scrapée"

  # ── API FastAPI ──────────────────────────────────────────────────────────
  - name: techshop-api
    rules:
      - alert: APITauxErreurEleve
        expr: job:http_requests_errors:ratio_rate5m{job="techshop-api"} > 0.05
        for: 5m
        labels:
          severity: critical
          team: backend
        annotations:
          summary: "Plus de 5 % de réponses 5xx sur l'API"
          description: "Taux d'erreur actuel : {{ $value | humanizePercentage }}."
          runbook_url: "https://wiki.techshop.local/runbooks/api-erreurs"

      - alert: APILatenceElevee
        expr: job:http_request_duration_seconds:p95_5m{job="techshop-api"} > 0.5
        for: 10m
        labels:
          severity: warning
          team: backend
        annotations:
          summary: "p95 de latence API > 500 ms"
          description: "p95 actuel : {{ $value | humanizeDuration }}."

      - alert: APISansTrafic
        expr: job:http_requests:rate5m{job="techshop-api"} == 0
        for: 15m
        labels:
          severity: warning
          team: backend
        annotations:
          summary: "L'API ne reçoit plus aucune requête depuis 15 min"

  # ── Modèle ML ────────────────────────────────────────────────────────────
  - name: techshop-ml
    rules:
      - alert: ModeleNonPret
        expr: techshop_modele_etat{techshop_modele_etat="pret"} == 0
        for: 2m
        labels:
          severity: critical
          team: data
        annotations:
          summary: "Le modèle n'est pas dans l'état 'pret' sur {{ $labels.instance }}"

      - alert: InferenceLente
        expr: modele:techshop_modele_inference_seconds:p95_5m > 0.25
        for: 10m
        labels:
          severity: warning
          team: data
        annotations:
          summary: "Inférence lente pour {{ $labels.modele }}"
          description: "p95 de predict_proba : {{ $value | humanizeDuration }}."

      - alert: ErreursPrediction
        expr: techshop_prediction_erreurs:rate5m > 0.1
        for: 5m
        labels:
          severity: warning
          team: data
        annotations:
          summary: "Plus de 0,1 erreur de prédiction / seconde"

      - alert: DeriveDesDonnees
        expr: feature:techshop_feature_drift:ratio > 0.15
        for: 1h
        labels:
          severity: warning
          team: data
        annotations:
          summary: "Drift sur la feature {{ $labels.feature }}"
          description: "Écart à la moyenne d'entraînement : {{ $value | humanizePercentage }}. Envisager un réentraînement."

      - alert: ModeleTropAncien
        expr: time() - techshop_modele_dernier_chargement_timestamp_seconds > 30 * 24 * 3600
        labels:
          severity: info
          team: data
        annotations:
          summary: "Modèle chargé il y a plus de 30 jours"

  # ── Métier ───────────────────────────────────────────────────────────────
  - name: techshop-metier
    rules:
      - alert: TauxRefusCommandesEleve
        expr: techshop_commandes_refusees:ratio_rate15m > 0.2
        for: 15m
        labels:
          severity: warning
          team: metier
        annotations:
          summary: "Plus de 20 % de commandes refusées"
          description: "Taux de refus : {{ $value | humanizePercentage }}."

      - alert: APIExterneTimeouts
        expr: api:techshop_api_externe_timeouts:ratio_rate5m > 0.1
        for: 5m
        labels:
          severity: warning
          team: backend
        annotations:
          summary: "API partenaire {{ $labels.api }} : plus de 10 % de timeouts"

      - alert: StockBas
        expr: techshop_stock_produits < 5
        for: 30m
        labels:
          severity: info
          team: metier
        annotations:
          summary: "Stock bas : {{ $labels.categorie }} ({{ $value }} unités)"

      - alert: ETLEnRetard
        expr: time() - techshop_etl_derniere_reussite_timestamp_seconds > 26 * 3600
        labels:
          severity: critical
          team: data
        annotations:
          summary: "L'ETL de nuit n'a pas réussi depuis plus de 26 h"

  # ── Infrastructure ───────────────────────────────────────────────────────
  - name: techshop-infra
    rules:
      - alert: CPUSature
        expr: instance:node_cpu_utilisation:rate5m > 0.9
        for: 10m
        labels:
          severity: warning
          team: ops
        annotations:
          summary: "CPU > 90 % sur {{ $labels.instance }}"

      - alert: MemoireFaible
        expr: instance:node_memory_disponible:ratio < 0.1
        for: 5m
        labels:
          severity: critical
          team: ops
        annotations:
          summary: "Moins de 10 % de RAM disponible sur {{ $labels.instance }}"

      - alert: DisquePresquePlein
        expr: instance_mountpoint:node_filesystem_libre:ratio < 0.1
        for: 5m
        labels:
          severity: critical
          team: ops
        annotations:
          summary: "{{ $labels.mountpoint }} : moins de 10 % d'espace libre"

      - alert: DisquePleinDans24h          # prédiction linéaire sur les 6 dernières heures
        expr: predict_linear(node_filesystem_avail_bytes{fstype!~"tmpfs|overlay|squashfs"}[6h], 24 * 3600) < 0
        for: 30m
        labels:
          severity: warning
          team: ops
        annotations:
          summary: "{{ $labels.mountpoint }} sera plein dans moins de 24 h au rythme actuel"

      - alert: MySQLDown
        expr: mysql_up == 0
        for: 1m
        labels:
          severity: critical
          team: ops
        annotations:
          summary: "MySQL injoignable depuis mysqld_exporter"

      - alert: MySQLTropDeConnexions
        expr: instance:mysql_connexions:ratio > 0.8
        for: 5m
        labels:
          severity: warning
          team: ops
        annotations:
          summary: "MySQL utilise plus de 80 % de max_connections"
"""
print(alert_rules_yaml)

In [ ]:
# 13.6 — Alertmanager (config/alertmanager/alertmanager.yml)

alertmanager_yaml = """
# config/alertmanager/alertmanager.yml
# Alertmanager reçoit les alertes "firing" de Prometheus et décide :
#   - comment les REGROUPER (une notification pour plusieurs alertes liées)
#   - à QUI les envoyer (routes → receivers)
#   - quand les TAIRE (inhibitions, silences, plages horaires)

global:
  resolve_timeout: 5m                       # alerte considérée résolue si plus reçue depuis 5 min
  smtp_smarthost: "smtp.techshop.local:587"
  smtp_from: "alertmanager@techshop.local"
  smtp_auth_username: "alertmanager@techshop.local"
  smtp_auth_password_file: /etc/alertmanager/secrets/smtp_password   # jamais en clair dans le YAML
  smtp_require_tls: true

# ── Routage : arbre de routes, la première route qui correspond gagne ──────
route:
  receiver: equipe-ops-email               # receveur par défaut (si aucune sous-route ne correspond)
  group_by: [alertname, job]               # une notification par (alertname, job)
  group_wait: 30s                          # attendre avant le 1er envoi d'un groupe (regroupe les rafales)
  group_interval: 5m                       # délai avant de renvoyer un groupe qui a changé
  repeat_interval: 4h                      # relance tant que l'alerte est toujours active

  routes:
    # Alertes "info" : seulement journalisées par le webhook (pas de notification humaine)
    - matchers:
        - severity="info"
      receiver: webhook-journal

    # Toute alerte critique → Slack #alertes-prod, en plus de la suite (continue: true)
    - matchers:
        - severity="critical"
      receiver: slack-critique
      group_wait: 10s
      repeat_interval: 1h
      continue: true

    # Alertes de l'équipe data (ML, ETL)
    - matchers:
        - team="data"
      receiver: equipe-data
      routes:
        - matchers:
            - alertname="DeriveDesDonnees"
          receiver: equipe-data
          group_by: [alertname, feature]
          repeat_interval: 24h               # un drift ne justifie pas une relance toutes les 4 h

    # Alertes métier : pas la nuit ni le week-end
    - matchers:
        - team="metier"
      receiver: equipe-metier-email
      active_time_intervals: [heures-ouvrees]

# ── Inhibitions : une alerte en masque d'autres (évite le bruit) ───────────
inhibit_rules:
  # Une critique masque la warning du même nom sur la même instance
  - source_matchers: [severity="critical"]
    target_matchers: [severity="warning"]
    equal: [alertname, instance]

  # Si une cible est injoignable, inutile d'alerter sur ses métriques
  - source_matchers: [alertname="CibleInjoignable"]
    target_matchers: [severity=~"warning|info"]
    equal: [instance]

  # MySQL down → masque les alertes de connexions MySQL
  - source_matchers: [alertname="MySQLDown"]
    target_matchers: [alertname="MySQLTropDeConnexions"]
    equal: [instance]

  # Modèle pas prêt → l'inférence lente et les erreurs de prédiction en découlent
  - source_matchers: [alertname="ModeleNonPret"]
    target_matchers: [alertname=~"InferenceLente|ErreursPrediction"]

# ── Plages horaires ────────────────────────────────────────────────────────
time_intervals:
  - name: heures-ouvrees
    time_intervals:
      - weekdays: ["monday:friday"]
        times:
          - start_time: "08:00"
            end_time: "19:00"
        location: "Europe/Paris"

# ── Receveurs : où partent les notifications ───────────────────────────────
receivers:
  - name: equipe-ops-email
    email_configs:
      - to: "ops@techshop.local"
        send_resolved: true

  - name: slack-critique
    slack_configs:
      - api_url_file: /etc/alertmanager/secrets/slack_webhook_url
        channel: "#alertes-prod"
        send_resolved: true
        title: '[{{ .Status | toUpper }}] {{ .CommonLabels.alertname }} ({{ .Alerts | len }})'
        text: |-
          {{ range .Alerts }}• *{{ .Annotations.summary }}*
          {{ if .Annotations.description }}{{ .Annotations.description }}
          {{ end }}{{ if .Annotations.runbook_url }}Runbook : {{ .Annotations.runbook_url }}
          {{ end }}{{ end }}

  - name: equipe-data
    email_configs:
      - to: "data@techshop.local"
        send_resolved: true
    slack_configs:
      - api_url_file: /etc/alertmanager/secrets/slack_webhook_url
        channel: "#data-ml"
        send_resolved: true

  - name: equipe-metier-email
    email_configs:
      - to: "metier@techshop.local"

  - name: webhook-journal
    webhook_configs:
      - url: "http://api:8000/alertes/webhook"     # endpoint de l'API qui journalise (JSON)
        send_resolved: true
"""
print(alertmanager_yaml)

print("""
=== Secrets (référencés par *_file, jamais écrits dans le YAML) ===
mkdir -p config/alertmanager/secrets
printf '%s' 'mot_de_passe_smtp'                         > config/alertmanager/secrets/smtp_password
printf '%s' 'https://hooks.slack.com/services/T0/B0/XX' > config/alertmanager/secrets/slack_webhook_url
echo 'config/alertmanager/secrets/' >> .gitignore

=== Routage : quelle alerte va où ? (amtool config routes test) ===
severity=critical team=backend alertname=APITauxErreurEleve      → slack-critique
severity=warning  team=data    alertname=DeriveDesDonnees        → equipe-data
severity=info     team=metier  alertname=StockBas                → webhook-journal
severity=warning  team=metier  alertname=TauxRefusCommandesEleve → equipe-metier-email (8h-19h en semaine)
severity=warning  team=ops     alertname=CPUSature               → equipe-ops-email (receveur par défaut)

=== À retenir ===
group_by / group_wait   → 1 notification pour une rafale d'alertes liées
repeat_interval         → relance si l'alerte est toujours active
continue: true          → l'alerte continue vers les routes suivantes
inhibit_rules           → une alerte en masque d'autres (cible down → pas d'alerte CPU dessus)
silences                → mise en sourdine temporaire (maintenance), via l'UI ou amtool
""")

In [ ]:
# 13.7 — Tester les règles et piloter l'alerting

rules_test_yaml = """
# config/rules/tests/rules_test.yml
# Tests unitaires des règles : on INJECTE des séries fictives et on vérifie
# ce que les règles produisent.  →  promtool test rules rules/tests/rules_test.yml

rule_files:
  - ../recording_rules.yml
  - ../alert_rules.yml

evaluation_interval: 1m

tests:

  # 1) Règle d'enregistrement : taux d'erreur 5xx de l'API
  - interval: 1m
    input_series:
      # notation a+bxN : part de a, ajoute b à chaque pas, N fois
      - series: 'http_requests_total{job="techshop-api", handler="/predict", method="POST", status="2xx"}'
        values: '0+90x20'               # 90 requêtes OK / min
      - series: 'http_requests_total{job="techshop-api", handler="/predict", method="POST", status="5xx"}'
        values: '0+10x20'               # 10 erreurs / min  → 10 %
    promql_expr_test:
      - expr: job:http_requests_errors:ratio_rate5m
        eval_time: 10m
        exp_samples:
          - labels: 'job:http_requests_errors:ratio_rate5m{job="techshop-api"}'
            value: 0.1

  # 2) Alerte : 10 % d'erreurs > seuil de 5 % → pending puis firing après 5 min
  - interval: 1m
    input_series:
      - series: 'http_requests_total{job="techshop-api", handler="/predict", method="POST", status="2xx"}'
        values: '0+90x20'
      - series: 'http_requests_total{job="techshop-api", handler="/predict", method="POST", status="5xx"}'
        values: '0+10x20'
    alert_rule_test:
      - eval_time: 3m                   # condition vraie depuis peu → encore "pending"
        alertname: APITauxErreurEleve
        exp_alerts: []
      - eval_time: 12m                  # vraie depuis plus de 5 min → "firing"
        alertname: APITauxErreurEleve
        exp_alerts:
          - exp_labels:
              severity: critical
              team: backend
              job: techshop-api
            exp_annotations:
              summary: "Plus de 5 % de réponses 5xx sur l'API"
              description: "Taux d'erreur actuel : 10%."
              runbook_url: "https://wiki.techshop.local/runbooks/api-erreurs"

  # 3) Alerte : cible injoignable
  - interval: 1m
    input_series:
      - series: 'up{job="mysql", instance="mysql-exporter:9104"}'
        values: '1 1 0 0 0 0'
    alert_rule_test:
      - eval_time: 5m
        alertname: CibleInjoignable
        exp_alerts:
          - exp_labels:
              severity: critical
              team: ops
              job: mysql
              instance: mysql-exporter:9104
            exp_annotations:
              summary: "mysql injoignable (mysql-exporter:9104)"
              description: "Prometheus n'arrive plus à scraper mysql-exporter:9104 depuis plus d'une minute."

  # 4) Drift : moyenne prod de l'âge = 48 vs 40.2 à l'entraînement → +19 % > 15 %
  # (un point par minute : au-delà de 5 min sans point, une série devient "stale")
  - interval: 1m
    input_series:
      - series: 'techshop_feature_moyenne{feature="age", instance="api:8000", job="techshop-api"}'
        values: '48x90'
    alert_rule_test:
      - eval_time: 30m                  # condition vraie, mais for: 1h pas encore atteint
        alertname: DeriveDesDonnees
        exp_alerts: []
      - eval_time: 75m
        alertname: DeriveDesDonnees
        exp_alerts:
          - exp_labels:
              severity: warning
              team: data
              feature: age
            exp_annotations:
              summary: "Drift sur la feature age"
              description: "Écart à la moyenne d'entraînement : 19.4%. Envisager un réentraînement."
"""
print(rules_test_yaml)

print("""
=== Valider AVANT de déployer (images officielles, rien à installer) ===
# Config Prometheus + toutes les règles qu'elle référence
docker run --rm -v ./config:/etc/prometheus --entrypoint promtool \\
  prom/prometheus:v2.54.1 check config /etc/prometheus/prometheus.yml

# Tests unitaires des règles
docker run --rm -v ./config:/etc/prometheus -w /etc/prometheus/rules/tests --entrypoint promtool \\
  prom/prometheus:v2.54.1 test rules rules_test.yml

# Config Alertmanager + test du routage
docker run --rm -v ./config/alertmanager:/etc/alertmanager --entrypoint amtool \\
  prom/alertmanager:v0.27.0 check-config /etc/alertmanager/alertmanager.yml
docker run --rm -v ./config/alertmanager:/etc/alertmanager --entrypoint amtool \\
  prom/alertmanager:v0.27.0 config routes test --config.file=/etc/alertmanager/alertmanager.yml \\
  severity=critical team=backend alertname=APITauxErreurEleve

=== Recharger après modification (sans redémarrer) ===
curl -X POST http://localhost:9090/-/reload      # Prometheus (--web.enable-lifecycle)
curl -X POST http://localhost:9093/-/reload      # Alertmanager

=== Observer ===
http://localhost:9090/rules     → règles chargées, dernière évaluation, erreurs
http://localhost:9090/alerts    → inactive / pending / firing
http://localhost:9093           → alertes reçues, groupes, silences
ALERTS{alertstate=\"firing\"}      → requête PromQL : alertes actives

=== Déclencher une fausse alerte (tester les notifications) ===
docker compose exec alertmanager amtool alert add alertname=AlerteTest severity=warning team=ops \\
  --annotation=summary=\"Test de notification\" --alertmanager.url=http://localhost:9093

=== Silence pendant une maintenance ===
docker compose exec alertmanager amtool silence add alertname=CPUSature --duration=2h \\
  --comment=\"maintenance serveur\" --alertmanager.url=http://localhost:9093
docker compose exec alertmanager amtool silence query --alertmanager.url=http://localhost:9093
""")

---
<a id="14-grafana"></a>
## 14. Grafana — Datasources & Dashboards

### Provisioning automatique (Infrastructure as Code)
Grafana peut être configuré via des fichiers YAML au lieu de l'UI.
Ces fichiers sont montés dans `/etc/grafana/provisioning/`.

```
/etc/grafana/provisioning/
├── datasources/
│   └── prometheus.yml      → déclaration de la source Prometheus
└── dashboards/
    ├── dashboard.yml       → chemin des fichiers JSON
    └── techshop.json       → dashboard exporté
```


In [ ]:
# 14.1 — Datasource Prometheus via fichier YAML

datasource_yaml = """
# config/grafana/datasources/prometheus.yml
# Ce fichier est monté dans /etc/grafana/provisioning/datasources/

apiVersion: 1

datasources:
  - name: Prometheus
    type: prometheus
    access: proxy              # Grafana proxy les requêtes vers Prometheus
    orgId: 1
    url: http://prometheus:9090   # URL interne Docker/K8s
    basicAuth: false
    isDefault: true
    version: 1
    editable: false            # false = non modifiable depuis l'UI

  - name: Prometheus-Dev
    type: prometheus
    url: http://prometheus-dev:9090
    isDefault: false
    jsonData:
      timeInterval: "15s"      # résolution minimale
      queryTimeout: "60s"
      httpMethod: POST
"""
print("datasources/prometheus.yml :")
print(datasource_yaml)

dashboard_provision = """
# config/grafana/dashboards/dashboard.yml

apiVersion: 1

providers:
  - name: TechShop Dashboards
    orgId: 1
    type: file
    disableDeletion: false
    updateIntervalSeconds: 10
    options:
      path: /etc/grafana/provisioning/dashboards   # dossier contenant les .json
      foldersFromFilesStructure: true
"""
print("\ndashboards/dashboard.yml :")
print(dashboard_provision)

In [ ]:
# 14.2 — Créer un dashboard depuis l'UI (guide étape par étape)

print("""
=== Créer un dashboard Grafana — étapes ===

1. Accéder à Grafana : http://localhost:3000
   Login : admin / admin (changer au premier login)

2. Ajouter la datasource (si pas provisionnée) :
   Configuration (⚙) → Data Sources → Add data source
   → Prometheus → URL: http://prometheus:9090 → Save & Test

3. Créer un nouveau dashboard :
   + (Create) → New Dashboard → Add a new panel

4. Configurer un panneau (Panel) :
   - Metrics browser : saisir une requête PromQL
     ex: rate(http_requests_total[5m])
   - Choisir le type de visualisation : Time series, Stat, Gauge, Bar chart...
   - Panel title : "Taux de requêtes/sec"

5. Exemples de panels utiles pour une API ML :
   ┌───────────────────────────────────────────────────────────┐
   │ Panel : Requêtes/sec                                      │
   │ Query : sum(rate(http_requests_total[5m]))                │
   │ Type  : Time series                                       │
   ├───────────────────────────────────────────────────────────┤
   │ Panel : Latence p95                                       │
   │ Query : histogram_quantile(0.95,                          │
   │           sum(rate(http_request_duration_seconds_bucket   │
   │                    [5m])) by (le))                        │
   │ Type  : Time series, Unit: seconds                        │
   ├───────────────────────────────────────────────────────────┤
   │ Panel : Taux d'erreur (%)                                 │
   │ Query : 100 * sum(rate(http_requests_total                │
   │           {status_code=~"5.."}[5m]))                      │
   │         / sum(rate(http_requests_total[5m]))              │
   │ Type  : Stat, Thresholds: 1%=yellow, 5%=red              │
   ├───────────────────────────────────────────────────────────┤
   │ Panel : Prédictions de churn par risque                   │
   │ Query : sum by(risque)                                    │
   │           (rate(techshop_predictions_total[1m]))          │
   │ Type  : Bar chart                                         │
   └───────────────────────────────────────────────────────────┘

6. Sauvegarder le dashboard : icône 💾 → Title → Save
7. Exporter le JSON : Share → Export → Save to file
   (à versionner dans Git pour le provisioning automatique)

8. Variables de dashboard (pour filtrer dynamiquement) :
   Dashboard Settings → Variables → Add variable
   Type : Query, Query : label_values(http_requests_total, handler)
   Utilisation dans les panels : {handler="$handler"}
""")

---
## Cheat Sheet finale

| Sujet | Commande / Snippet clé |
|-------|----------------------|
| Variables bash | `export DB_HOST=localhost` → `echo $DB_HOST` |
| .bashrc | `echo 'export VAR=val' >> ~/.bashrc && source ~/.bashrc` |
| venv | `python -m venv .venv && source .venv/bin/activate` |
| dotenv Python | `from dotenv import load_dotenv; load_dotenv(".env")` |
| requests GET | `r = requests.get(url, params={...}, timeout=10); r.raise_for_status()` |
| curl POST JSON | `curl -s --json '{...}' http://localhost:8000/predict` |
| joblib save | `joblib.dump(model, "model.joblib", compress=3)` |
| joblib load | `model = joblib.load("model.joblib")` |
| FastAPI démarrer | `uvicorn main:app --host 0.0.0.0 --port 8000 --reload` |
| Prometheus expose | `Instrumentator().instrument(app).expose(app)` |
| pytest | `pytest tests/ -v --tb=short --cov=src` |
| kubectl apply | `kubectl apply -f fichier.yaml` |
| kubectl get all | `kubectl get all -n mon-namespace` |
| kubectl logs | `kubectl logs -f -l app=mon-app -n mon-ns` |
| kubectl exec | `kubectl exec -it pod/MON_POD -n mon-ns -- bash` |
| port-forward | `kubectl port-forward service/api 8080:80 -n mon-ns` |
| minikube start | `minikube start --driver=docker --cpus=2 --memory=4096` |
| GitLab CI trigger | `git push origin main` (pipeline auto) |
| Runner register | `sudo gitlab-runner register --executor docker` |
| Docker build+push | `docker build -t compte/app:v1 . && docker push compte/app:v1` |
| PromQL taux | `rate(http_requests_total[5m])` |
| mysqld_exporter | `prom/mysqld-exporter` sur `:9104`, `--mysqld.address=db:3306` + `MYSQLD_EXPORTER_PASSWORD` |
| Droits exporter | `GRANT PROCESS, REPLICATION CLIENT, SELECT ON *.* TO 'exporter'@'%';` |
| PromQL MySQL | `mysql_up`, `rate(mysql_global_status_queries[5m])` |
| Métrique custom | `Counter(name="techshop_x_total", documentation="...", labelnames=["canal"]).labels(canal="web").inc()` |
| Chronométrer | `with HIST.time():` / `@SUMMARY.time()` |
| Job batch | `push_to_gateway(gateway="pushgateway:9091", job="etl", registry=registry)` |
| PromQL p95 | `histogram_quantile(0.95, sum(rate(http_request_duration_seconds_bucket[5m])) by (le))` |
| Recording rule | `record: job:http_requests:rate5m` + `expr: sum by (job) (rate(http_requests_total[5m]))` |
| Alerte | `alert: X` + `expr: ... > seuil` + `for: 5m` + `labels: {severity: critical}` |
| Vérifier les règles | `promtool check config prometheus.yml` / `promtool test rules rules_test.yml` |
| Alertmanager | `route` (group_by, receiver, matchers) → `receivers` ; `inhibit_rules` ; `amtool check-config` |
| Recharger | `curl -X POST localhost:9090/-/reload` (avec `--web.enable-lifecycle`) |
| Grafana URL | `http://localhost:3000` (admin/admin) |

---
> **Bonne chance pour l'examen !**
